## DS50 Project — LLM-as-a-Judge for RAG

##### **Objective :** Build a Retrieval-Augmented Generation (RAG) pipeline, test multiple open-source LLMs, and evaluate them using an LLM-as-a-Judge approach.

  --------------------------------------------------------------------------------------------------------------------------------------------------------

### **PHASE 1: RAG PIPELINE**

**Framework** : LangChain<br>
**Dataset** : PubMedQA — pqa_artificial subset<br>
**Output** : results/RAGOutput/combined_rag_results.csv

All generator LLMs run via the **NVIDIA Build API** — no local loading, no GPU, no Hugging Face login, no quantization.

**Generator models**
- llama_3b — meta/llama-3.2-3b-instruct
- llama_8b — meta/llama-3.1-8b-instruct
- llama_70b — meta/llama-3.3-70b-instruct
- mistral_small — mistralai/mistral-small-4-119b-2603
- qwen_397b — qwen/qwen3.5-397b-a17b

**Embedding** : nvidia/llama-nemotron-embed-1b-v2<br>
**Vector store** : FAISS (local, CPU)

In [ ]:
# 1 — INSTALLATION
# faiss-cpu              → vector store for retrieval
# datasets               → loads PubMedQA from HuggingFace
# pandas                 → handles results DataFrames
# matplotlib             → plotting (Phase 3 dashboard)
# tqdm                   → progress bars
# langchain              → RAG pipeline orchestration
# langchain-core         → Document, PromptTemplate
# langchain-community    → FAISS wrapper for LangChain
# langchain-text-splitters → RecursiveCharacterTextSplitter
# langchain-nvidia-ai-endpoints → ChatNVIDIA for NVIDIA Build API

!pip install -q \
  faiss-cpu \
  datasets \
  pandas \
  matplotlib \
  tqdm \
  langchain \
  langchain-core \
  langchain-community \
  langchain-text-splitters \
  langchain-nvidia-ai-endpoints

In [ ]:
# 2 — IMPORTS

import os
import time
import json
import getpass
import pandas as pd
from tqdm import tqdm

from datasets import load_dataset

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_nvidia_ai_endpoints import (
    ChatNVIDIA,
    NVIDIAEmbeddings,
)

print("All imports successful.")

In [ ]:
# 3 — NVIDIA API KEY

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA API key: ")

if os.environ["NVIDIA_API_KEY"].startswith("nvapi-"):
    print("NVIDIA API key loaded.")
else:
    print("Warning: key may be invalid — check before running.")

In [ ]:
# 4 — CONFIGURATION
# All parameters centralized here.
# Change values here only — never scattered across the notebook.

CONFIG = {
    # Dataset
    "dataset_name"    : "qiaojin/PubMedQA",
    "dataset_subset"  : "pqa_artificial",
    "num_docs"        : 200,   # PubMedQA items used to build the corpus
    "num_questions"   : 50,    # questions evaluated per model

    # Retrieval
    "chunk_size"      : 512,   # max characters per chunk
    "chunk_overlap"   : 64,    # overlap between consecutive chunks
    "top_k"        : 5,        # top-k chunks retrieved from FAISS

    # Embedding model — NVIDIA API
    # llama-nemotron-embed-1b-v2 runs on NVIDIA servers
    # Specialized for semantic similarity and retrieval tasks
    "embedding_model" : "nvidia/llama-nemotron-embed-1b-v2",

    # Generator LLMs — 5 models, one per group member
    "generator_models": {
        "llama_3b"    : "meta/llama-3.2-3b-instruct",
        "llama_8b"    : "meta/llama-3.1-8b-instruct",        
        "llama_70b"   : "meta/llama-3.3-70b-instruct",
        "mistral_small" : "mistralai/mistral-small-4-119b-2603",
        "qwen_397b"   : "qwen/qwen3.5-397b-a17b",
    },

    # Generation parameters
    "temperature"     : 0.0,              # deterministic output — reproducible results
    "max_completion_tokens"      : 512,   # 5-8 sentence biomedical answers

    # Output
    "results_dir"     : "results",
    "rag_output_dir" : os.path.join("results", "RAGOutput"),
}

os.makedirs(CONFIG["results_dir"], exist_ok=True)
os.makedirs(CONFIG["rag_output_dir"], exist_ok=True) 
print("Configuration ready.")
print(f"   Dataset    : {CONFIG['dataset_name']} / {CONFIG['dataset_subset']}")
print(f"   Questions  : {CONFIG['num_questions']} per model")
print(f"   Embedding : {CONFIG['embedding_model']}")
print(f"   top_k     : {CONFIG['top_k']} chunks (FAISS direct)")
print(f"   Models    :")
for key, name in CONFIG["generator_models"].items():
    print(f"      {key:20s} → {name}")
print(f"   Results at : {CONFIG['results_dir']}/")

In [ ]:
# 5 — CONTEXT BUILDER
# Concatenates retrieved chunks into one plain text string.
# This string is injected into the RAG prompt as the context.

def build_context_from_docs(docs: list) -> str:
    """
    Takes a list of LangChain Document objects and returns
    a single string with all chunk texts separated by blank lines.
    This is what gets passed to the LLM as the retrieved context.
    """
    return "\n\n".join([doc.page_content for doc in docs])

In [ ]:
# 6 — RAG INFERENCE FUNCTION
# This function runs the full RAG pipeline for ONE question using ONE model.
# It is called 50 times per model in the experiment loop.

def ask_rag(question: str,
            retriever,
            llm,
            prompt_template: PromptTemplate,
            retries: int = 3,
            wait: int = 10) -> dict:
    """
    Runs the full RAG pipeline for a single question:
      1. Retrieve top-k relevant chunks from FAISS
      2. Build context string from retrieved chunks
      3. Format the RAG prompt with context + question
      4. Call the generator LLM via NVIDIA API
      5. Return structured result

    Includes retry logic: if the API returns an error or empty response,
    it waits and retries up to `retries` times before giving up.

    Args:
        question        : the PubMedQA question
        retriever       : FAISS retriever object
        llm             : ChatNVIDIA model instance
        prompt_template : the RAG PromptTemplate
        retries         : max number of API retry attempts
        wait            : seconds to wait between retries

    Returns:
        dict with keys: question, generated_answer,
                        source_documents, context
    """
    # Step 1 — Retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)

    # Step 2 — Build context string
    context = build_context_from_docs(retrieved_docs)

    # Step 3 — Format prompt
    final_prompt = prompt_template.format(
        context  = context,
        question = question,
    )

    # Step 4 — Call LLM with retry logic
    for attempt in range(1, retries + 1):
        try:
            response         = llm.invoke(final_prompt)
            generated_answer = response.content.strip()

            # Guard against empty API responses
            if not generated_answer:
                raise ValueError("API returned empty response")
            break

        except Exception as e:
            if attempt < retries:
                print(f"     Attempt {attempt}/{retries} failed: {e}. "
                      f"Waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"     All {retries} attempts failed.")
                generated_answer = f"[ERROR] {str(e)}"

    return {
        "question"        : question,
        "generated_answer": generated_answer,
        "source_documents": retrieved_docs,
        "context"         : context,
    }


In [ ]:
# 7 — DATASET LOADING
# Loads PubMedQA pqa_artificial from HuggingFace.
# No HuggingFace login needed — pqa_artificial is publicly available.

# Each item contains:
#   question       - the medical research question
#   context        - list of abstract sentences (retrieval corpus)
#   long_answer    - detailed reference answer
#   final_decision - yes/no/maybe (NOT used — we evaluate long answers)

print("Loading PubMedQA dataset...")
dataset    = load_dataset(CONFIG["dataset_name"], CONFIG["dataset_subset"])
train_data = dataset["train"]

print(f" Loaded: {len(train_data)} examples")
print(f"   Columns: {train_data.column_names}")
print(f"\n--- Sample (index 0) ---")
print(f"   Question    : {train_data[0]['question']}")
print(f"   Long answer : {train_data[0]['long_answer'][:150]}...")
print(f"   contexts  : {len(train_data[0]['context']['contexts'])}")

In [ ]:
# 8 — DOCUMENT PREPARATION
# Extracts context passages from PubMedQA items and wraps them in LangChain Document objects with metadata.

# Only context.contexts (the real text) goes into FAISS.
# Labels and MeSH terms are stored as metadata for traceability but never influence retrieval or generation.

print("\nPreparing documents from corpus...")
documents = []

for i in range(min(CONFIG["num_docs"], len(train_data))):
    item     = train_data[i]
    contexts = item["context"]["contexts"]
    labels   = item["context"]["labels"]
    meshes   = item["context"]["meshes"]

    for j, ctx in enumerate(contexts):
        doc = Document(
            page_content = ctx,
            metadata     = {
                "source_id"    : i,
                "chunk_id"     : j,
                "question"     : item["question"],
                "context_label": labels[j] if j < len(labels) else None,
                "mesh_terms"   : meshes[j] if j < len(meshes) else None,
            },
        )
        documents.append(doc)

print(f"   {len(documents)} raw passages prepared.")
print(f"   Example: {documents[0].page_content[:150]}...")

In [ ]:
# 9 — CHUNKING
# Splits passages into smaller chunks for more precise retrieval.
#
# chunk_size=512   → max characters per chunk
# chunk_overlap=64 → 64 chars shared between consecutive chunks prevents cutting a sentence at a boundary

print("\nChunking documents...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = CONFIG["chunk_size"],
    chunk_overlap = CONFIG["chunk_overlap"],
)
split_docs = text_splitter.split_documents(documents)

print(f" {len(documents)} passages → {len(split_docs)} chunks.")

In [ ]:
# 10 — NVIDIA EMBEDDINGS + FAISS VECTOR STORE
# nvidia/llama-nemotron-embed-1b-v2 runs via NVIDIA API
# Specialized for retrieval and semantic similarity tasks
# Replaces local PubMedBERT — no CPU/RAM needed for embedding

print("\nInitializing NVIDIA embedding model...")
embedding_model = NVIDIAEmbeddings(
    model   = CONFIG["embedding_model"],
    api_key = os.environ["NVIDIA_API_KEY"],
)
print(f" Embedding model ready: {CONFIG['embedding_model']}")

print("\nBuilding FAISS vector store...")
print("   This calls NVIDIA API to embed all chunks — may take a few minutes...")
vectorstore = FAISS.from_documents(split_docs, embedding_model)
print(f" FAISS index built — {len(split_docs)} chunks indexed.")

In [ ]:
# TEST NVIDIA embedding model
print("Testing NVIDIA llama-nemotron embeddings...")

test_texts = [
    "ILC2s may drive nasal polyp formation in CRS",
    "ILC2 frequencies were associated with nasal polyps (P=0.002)",
    "The weather is sunny today in Paris",
]

embeddings_test = embedding_model.embed_documents(test_texts)

from numpy import dot
from numpy.linalg import norm

def cosine_sim(a, b):
    return dot(a, b) / (norm(a) * norm(b))

sim_biomedical = cosine_sim(embeddings_test[0], embeddings_test[1])
sim_unrelated  = cosine_sim(embeddings_test[0], embeddings_test[2])

print(f"\nSimilarity biomedical pair : {sim_biomedical:.4f}")
print(f"Similarity unrelated pair  : {sim_unrelated:.4f}")

if sim_biomedical > sim_unrelated:
    print("   NVIDIA embedding works correctly")
    print("   Biomedical sentences closer than unrelated ones")
else:
    print("   Check embedding model")

In [ ]:
# 11 — RETRIEVER + SANITY CHECK
# Wraps the FAISS vector store as a LangChain retriever.
# For each question, FAISS returns the top_k most similar chunks using vector similarity (cosine) on the NVIDIA embeddings

# 11 — RETRIEVER
print("\nBuilding retriever...")

retriever = vectorstore.as_retriever(
    search_kwargs={"k": CONFIG["top_k"]}
)
print(f" Retriever ready — top_k={CONFIG['top_k']}")

# Sanity check
print("\n--- Retrieval sanity check ---")
test_question  = train_data[0]["question"]
test_retrieved = retriever.invoke(test_question)

print(f"Question : {test_question}")
print(f"Retrieved: {len(test_retrieved)} chunks")
for i, doc in enumerate(test_retrieved, 1):
    print(f"\n  Chunk {i}: {doc.page_content[:120]}...")

In [ ]:
# 12 — RAG PROMPT TEMPLATE
# Designed for biomedical long-answer generation.
# Forces the model to stay grounded in the retrieved context.
# Asks for 5-8 sentences to match PubMedQA long_answer format.
# temperature=0.0 makes output deterministic and reproducible.

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an expert biomedical researcher \
specialized in interpreting medical literature.

Answer the following biomedical question based STRICTLY \
on the retrieved context below.

=== Retrieved Context ===
{context}

=== Question ===
{question}

=== Instructions ===
1. Start with a DIRECT and CLEAR answer to the question
   in the first sentence (yes/no/partially + why).

2. Support your answer using ALL relevant findings
   from the retrieved context — do not omit key facts,
   statistics, or p-values that are relevant.

3. Interpret the evidence — do not just describe it.
   Example: say "X may drive Y" not just "X is associated with Y".

4. Use precise biomedical terminology.

5. Keep your answer concise and focused — 4 to 6 sentences.
   Do NOT add any information beyond what the context provides.
   Do NOT start with "According to the context" or similar phrases.

=== Answer ==="""
)

In [ ]:
# 13 — API CONNECTIVITY TEST
print("\n--- API Connectivity Test ---")
print("Testing llama_3b on question 0...")

test_llm = ChatNVIDIA(
    model                 = CONFIG["generator_models"]["llama_3b"],
    temperature           = CONFIG["temperature"],
    max_completion_tokens = CONFIG["max_completion_tokens"],
)

test_result = ask_rag(
    question        = train_data[0]["question"],
    retriever       = retriever,
    llm             = test_llm,
    prompt_template = rag_prompt,
)

if test_result["generated_answer"].startswith("[ERROR]"):
    print(f" API test failed: {test_result['generated_answer']}")
else:
    print(f" API test passed.")
    print(f"\nQuestion : {test_result['question']}")
    print(f"\nAnswer   : {test_result['generated_answer'][:300]}...")
    print(f"\nChunks   : {len(test_result['source_documents'])} retrieved after reranking")

In [ ]:
# 14 — EXPERIMENT RUNNER
# Runs the full RAG pipeline for ONE model across all questions.

def run_rag_experiment(model_key: str,
                       model_name: str,
                       train_data,
                       retriever,
                       prompt_template: PromptTemplate,
                       num_questions: int) -> pd.DataFrame:
    """
    Runs the full RAG pipeline for one generator model via NVIDIA API.

    For each of the num_questions questions:
      1. Retrieve top-k chunks from FAISS
      2. Build context string
      3. Format RAG prompt
      4. Call generator LLM via NVIDIA API
      5. Record: question, ground truth, generated answer,
                 retrieved contexts, inference time

    Saves results to a CSV file and returns the DataFrame.

    Args:
        model_key     : short identifier (e.g. "qwen_7b")
        model_name    : full NVIDIA API model ID
        train_data    : PubMedQA dataset
        retriever     : FAISS retriever
        prompt_template: RAG PromptTemplate
        num_questions : number of questions to evaluate

    Returns:
        pd.DataFrame with one row per question
    """
    print(f"\n{'='*60}")
    print(f"  Model    : {model_key}")
    print(f"  API name : {model_name}")
    print(f"  Questions: {num_questions}")
    print(f"{'='*60}")

    # Initialize the NVIDIA API model
    llm = ChatNVIDIA(
        model       = model_name,
        temperature = CONFIG["temperature"],
        max_completion_tokens  = CONFIG["max_completion_tokens"],
    )

    model_results    = []
    experiment_start = time.time()

    for i in tqdm(range(num_questions), desc=f"  {model_key}"):
        item         = train_data[i]
        question     = item["question"]
        ground_truth = item["long_answer"]

        start  = time.time()
        output = ask_rag(
            question        = question,
            retriever       = retriever,
            llm             = llm,
            prompt_template = prompt_template,
        )
        elapsed = round(time.time() - start, 4)

        # Serialize retrieved chunks as JSON string for clean CSV storage
        # Parsed back with json.loads() in Phase 2
        retrieved_contexts_json = json.dumps(
            [doc.page_content for doc in output["source_documents"]]
        )

        model_results.append({
            "model_key"              : model_key,
            "model_name"             : model_name,
            "question_id"            : i,
            "question"               : question,
            "ground_truth"           : ground_truth,
            "generated_answer"       : output["generated_answer"],
            "retrieved_contexts_json": retrieved_contexts_json,
            "full_context"           : output["context"],
            "inference_time_sec"     : elapsed,
        })

    results_df   = pd.DataFrame(model_results)
    total_time   = round(time.time() - experiment_start, 1)
    error_count  = results_df["generated_answer"].str.startswith("[ERROR]").sum()

    # Save individual model results
    save_path = os.path.join(CONFIG["rag_output_dir"], f"{model_key}_results.csv")
    results_df.to_csv(save_path, index=False)

    print(f"\n   Done — {model_key}")
    print(f"     Avg latency : {results_df['inference_time_sec'].mean():.2f}s")
    print(f"     Total time  : {total_time}s")
    print(f"     Errors      : {error_count}/{num_questions}")
    print(f"     Saved to    : {save_path}")

    return results_df

In [ ]:
# 15 — RUN MODELS
# Each model runs in its own cell below.
# Reason: 5 group members run in parallel, each on a separate session.
# Run ONLY your assigned cell — do not run the others.
#
# Member 1 → llama_3b      → Cell A
# Member 2 → qwen_7b       → Cell B
# Member 3 → mistral_24b   → Cell C
# Member 4 → llama_70b     → Cell D
# Member 5 → qwen_397b     → Cell E
print(" Ready — run your assigned model cell below.")

In [ ]:
# MODEL 1: llama_3b — meta/llama-3.2-3b-instruct

MODEL_KEY  = "llama_3b"
MODEL_NAME = CONFIG["generator_models"][MODEL_KEY]

print(f"Running: {MODEL_KEY}")
print(f"Model  : {MODEL_NAME}")
print(f"Questions: {CONFIG['num_questions']}")

df_llama_3b = run_rag_experiment(
    model_key       = MODEL_KEY,
    model_name      = MODEL_NAME,
    train_data      = train_data,
    retriever       = retriever,
    prompt_template = rag_prompt,
    num_questions   = CONFIG["num_questions"],
)

print(f"\n Done — {MODEL_KEY}_results.csv downloaded.")

In [ ]:
#  MODEL 2: llama_8b — meta/llama-3.2-8b-instruct

MODEL_KEY  = "llama_8b"
MODEL_NAME = CONFIG["generator_models"][MODEL_KEY]

print(f"Running: {MODEL_KEY}")
print(f"Model  : {MODEL_NAME}")
print(f"Questions: {CONFIG['num_questions']}")

df_llama_8b = run_rag_experiment(
    model_key       = MODEL_KEY,
    model_name      = MODEL_NAME,
    train_data      = train_data,
    retriever       = retriever,
    prompt_template = rag_prompt,
    num_questions   = CONFIG["num_questions"],
)

print(f"\n Done — {MODEL_KEY}_results.csv downloaded.")

In [ ]:
# MODEL 4: llama_70b — meta/llama-3.3-70b-instruct

MODEL_KEY  = "llama_70b"
MODEL_NAME = CONFIG["generator_models"][MODEL_KEY]

print(f"Running: {MODEL_KEY}")
print(f"Model  : {MODEL_NAME}")
print(f"Questions: {CONFIG['num_questions']}")

df_llama_70b = run_rag_experiment(
    model_key       = MODEL_KEY,
    model_name      = MODEL_NAME,
    train_data      = train_data,
    retriever       = retriever,
    prompt_template = rag_prompt,
    num_questions   = CONFIG["num_questions"],
)

print(f"\n Done — {MODEL_KEY}_results.csv downloaded.")

In [ ]:
# MODEL 4: mistral_small — mistralai/mistral-small-4-119b-2603

MODEL_KEY  = "mistral_small"
MODEL_NAME = CONFIG["generator_models"][MODEL_KEY]

print(f"Running: {MODEL_KEY}")
print(f"Model  : {MODEL_NAME}")
print(f"Questions: {CONFIG['num_questions']}")

df_mistral_small = run_rag_experiment(
    model_key       = MODEL_KEY,
    model_name      = MODEL_NAME,
    train_data      = train_data,
    retriever       = retriever,
    prompt_template = rag_prompt,
    num_questions   = CONFIG["num_questions"],
)

print(f"\n Done — {MODEL_KEY}_results.csv downloaded.")

In [ ]:
# MODEL 5: qwen_397b — qwen/qwen3.5-397b-a17b

MODEL_KEY  = "qwen_397b"
MODEL_NAME = CONFIG["generator_models"][MODEL_KEY]

print(f"Running: {MODEL_KEY}")
print(f"Model  : {MODEL_NAME}")
print(f"Questions: {CONFIG['num_questions']}")

df_qwen_397b = run_rag_experiment(
    model_key       = MODEL_KEY,
    model_name      = MODEL_NAME,
    train_data      = train_data,
    retriever       = retriever,
    prompt_template = rag_prompt,
    num_questions   = CONFIG["num_questions"],
)

#from google.colab import files
#files.download(f"results/{MODEL_KEY}_results.csv")
print(f"\n Done — {MODEL_KEY}_results.csv downloaded.")

In [ ]:
# 16 — COMBINE ALL 5 MODEL RESULTS
# Upload all 5 CSV files manually to results/ folder first.
# Then run this cell to merge them.

print("\nCombining results from all 5 models...")

all_dfs = []
for model_key in CONFIG["generator_models"].keys():

    path = os.path.join(
        CONFIG["rag_output_dir"],
        f"{model_key}_results.csv"
    )

    if os.path.exists(path):
        df     = pd.read_csv(path)
        errors = df["generated_answer"].str.startswith("[ERROR]").sum()
        all_dfs.append(df)
        print(f" {model_key}: {len(df)} rows — {errors} errors")
    else:
        print(f"  Not found: {path}")

if all_dfs:
    combined_df   = pd.concat(all_dfs, ignore_index=True)
    combined_path = os.path.join(
        CONFIG["rag_output_dir"], "combined_rag_results.csv"
    )
    combined_df.to_csv(combined_path, index=False)

    print(f"\n Combined: {len(combined_df)} rows")
    print(f"   Models  : {combined_df['model_key'].unique().tolist()}")
    print(f"   Saved   : {combined_path}")
else:
    print(" No files found — run all 5 model cells first")

In [ ]:
# 17 — FINAL SUMMARY

print("\n" + "="*60)
print("  PHASE 1 — EXPERIMENT SUMMARY")
print("="*60)

summary = (
    combined_df
    .groupby("model_key")
    .agg(
        questions   = ("question_id", "count"),
        avg_latency = ("inference_time_sec", "mean"),
        total_time  = ("inference_time_sec", "sum"),
        errors      = ("generated_answer",
                       lambda x: x.str.startswith("[ERROR]").sum()),
    )
    .round(3)
    .reset_index()
)

print(summary.to_string(index=False))
print("\n Phase 1 complete.")
print("   Output files ready for Phase 2 (LLM-as-a-Judge):")
for key in CONFIG["generator_models"]:
    print(f"   → results/{key}_results.csv")
print(f"   → results/combined_rag_results.csv")


  PHASE 1 — EXPERIMENT SUMMARY
    model_key  questions  avg_latency  total_time  errors
     llama_3b         50        3.126     156.320       0
    llama_70b         50       44.307    2215.340       0
     llama_8b         50        3.215     160.728       0
mistral_small         50        2.932     146.588       0
    qwen_397b         50       13.802     690.088       0

 Phase 1 complete.
   Output files ready for Phase 2 (LLM-as-a-Judge):
   → results/llama_3b_results.csv
   → results/llama_8b_results.csv
   → results/llama_70b_results.csv
   → results/mistral_small_results.csv
   → results/qwen_397b_results.csv
   → results/combined_rag_results.csv

   Download these files before closing the session.


---

### **PHASE 2 — JUDGE BENCHMARK - RAGAS vs Custom prompts**

**Objective** : Compare three judge LLMs (Mistral-Large, GPT-OSS-120B, Qwen3.5-397B) before choosing the final judge for Phase 2B.<br>
**Method** : Generate quality triplets (good / medium / bad) → score with all three judges → measure ranking accuracy + Pearson correlation.

In [ ]:
# PHASE 2 — 1: INSTALLATION
# Ragas + NVIDIA endpoints + scipy (Pearson correlation) for the judge benchmark.

!pip install -q \
  nest_asyncio \
  ragas>=0.2.0 \
  langchain-nvidia-ai-endpoints \
  langchain-huggingface \
  sentence-transformers \
  scipy \
  matplotlib \
  pandas \
  tqdm

In [ ]:
# PHASE 2 — 2: IMPORTS

import os, re, json, asyncio, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
from dataclasses import dataclass, field
from scipy.stats import pearsonr
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()

from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import SingleTurnMetric, MetricType
from ragas.metrics import (
    Faithfulness,      # groundedness metric
    AnswerRelevancy,   # answer relevance metric
    ContextPrecision,  # context relevance metric
    ContextRecall,     # context coverage metric
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

print(" Phase 2 imports ready.")

In [ ]:
# PHASE 2 — 3: CONFIGURATION
CONFIG_JUDGE = {
    # Phase 1 results
    "combined_results_path"  : "results/combined_rag_results.csv",

    # Neutral model to generate triplets
    # Must NOT be one of the judge candidates
    "triplet_generator_model": "meta/llama-3.1-8b-instruct",

    # Judge A — Mistral Large
    "judge_A_model"          : "mistralai/mistral-large-3-675b-instruct-2512",
    "judge_A_label"          : "Mistral-Large-675B",

    # Judge B — OpenAI OSS 120B
    "judge_B_model"          : "openai/gpt-oss-120b",
    "judge_B_label"          : "GPT-OSS-120B",

    # Judge C — Qwen 397B MoE
    "judge_C_model"          : "qwen/qwen3.5-397b-a17b",
    "judge_C_label"          : "Qwen3.5-397B",

    "judge_temperature"           : 0.0,
    "judge_max_completion_tokens" : 3000,
    "judge_A_sleep_seconds"       : 30,   # ← Mistral : 10s entre appels (rate limit strict)
    "judge_sleep_seconds"         : 3,    # ← GPT/Qwen : 3s entre appels
    
    # Number of questions for benchmark
    "n_benchmark_questions"  : 15,

    # Embedding model
    "embedding_model"        : "nvidia/llama-nemotron-embed-1b-v2",

    "results_dir"            : "results",
}

os.makedirs(CONFIG_JUDGE["results_dir"], exist_ok=True)
print(" Config ready.")
print(f"   Judge A : {CONFIG_JUDGE['judge_A_label']}")
print(f"   Judge B : {CONFIG_JUDGE['judge_B_label']}")
print(f"   Judge C : {CONFIG_JUDGE['judge_C_label']}")
print(f"   Triplets: {CONFIG_JUDGE['n_benchmark_questions']} questions × 3 quality levels")

In [ ]:
# PHASE 2 — 4: LOAD pqa_labeled FOR TRIPLET GENERATION
# We use the pqa_labeled subset (expert-annotated) from PubMedQA.
# This gives higher-quality ground_truth than pqa_artificial,
# which makes the judge benchmark more reliable.
# The same retriever (FAISS + BGE) built in Phase 1 is reused here.

print("\nLoading pqa_labeled from HuggingFace...")
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

labeled_ds = load_dataset(
    "qiaojin/PubMedQA",
    "pqa_labeled",
    split="train",
    trust_remote_code=True,
)
print(f" pqa_labeled loaded — {len(labeled_ds)} items")

# Build a clean DataFrame with the fields we need for triplet generation
labeled_rows = []
for item in labeled_ds:
    passages     = item["context"]["contexts"]     # list of text passages
    ground_truth = item.get("long_answer", "")     # expert long answer

    if not passages or not ground_truth:
        continue

    labeled_rows.append({
        "question_id" : str(item["pubid"]),
        "question"    : item["question"],
        "passages"    : passages,
        "ground_truth": ground_truth,
    })

labeled_df = pd.DataFrame(labeled_rows)
print(f" {len(labeled_df)} usable items after filtering.")

# Build a dedicated FAISS index from pqa_labeled passages only
# so each question is searched against its own correct passages
print("\nBuilding dedicated FAISS index from pqa_labeled passages...")
p2b_documents = []
for row in labeled_df.head(CONFIG_JUDGE["n_benchmark_questions"]).itertuples():
    for j, passage in enumerate(row.passages):
        p2b_documents.append(Document(
            page_content = passage,
            metadata     = {"question_id": row.question_id, "chunk_id": j}
        ))
os.environ["NVIDIA_API_KEY"] = "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"

p2b_embeddings = NVIDIAEmbeddings(
    model   = CONFIG_JUDGE["embedding_model"],
    api_key = os.environ["NVIDIA_API_KEY"],
)
p2b_vectorstore = FAISS.from_documents(p2b_documents, p2b_embeddings)
p2b_retriever   = p2b_vectorstore.as_retriever(search_kwargs={"k": 5})
print(f" p2b FAISS built — {len(p2b_documents)} passages indexed.")

# Retrieve contexts using the dedicated retriever
print("\nRetrieving contexts via p2b retriever...")
retrieved = []
for _, row in tqdm(
    labeled_df.head(CONFIG_JUDGE["n_benchmark_questions"]).iterrows(),
    total=CONFIG_JUDGE["n_benchmark_questions"],
):
    retrieved.append(row["passages"][:5]) 

benchmark_questions = (
    labeled_df
    .head(CONFIG_JUDGE["n_benchmark_questions"])
    .copy()
    .reset_index(drop=True)
)
benchmark_questions["retrieved_contexts"] = retrieved

print(f"\n {len(benchmark_questions)} questions ready for triplet generation.")
print(f"   Source   : pqa_labeled (expert-annotated)")
print(f"   Contexts : retrieved via dedicated p2b FAISS (pqa_labeled passages only)")

In [ ]:
# PHASE 2 — 5: TRIPLET GENERATOR
# Generates 3 answers per question at 3 quality levels using a neutral LLM.
# - GOOD   : faithful, relevant, complete
# - MEDIUM : partially correct, some irrelevant content
# - BAD    : vague, off-topic, contradicts context

triplet_llm = ChatNVIDIA(
    model       = CONFIG_JUDGE["triplet_generator_model"],
    temperature = 0.4,   # slight variance for diversity
    max_completion_tokens = 512,
)
print(f" Triplet generator ready: {CONFIG_JUDGE['triplet_generator_model']}")


TRIPLET_PROMPT = """\
You are a biomedical expert assistant.

You will receive a medical question, the retrieved context passages, \
and the correct ground truth answer written by medical experts.
Your task is to generate exactly 3 answers of deliberately different quality levels.

Question: {question}

Retrieved Context:
{context}

Ground Truth Answer (use only as reference for completeness):
{long_answer}

Generate the 3 answers following these strict instructions:

1. GOOD answer: Write a complete answer based STRICTLY on the retrieved context above.
   Every claim must be directly supported by the context.
   Use the ground truth only to ensure completeness — do NOT add facts not in the context.

2. MEDIUM answer: Write an answer based mostly on the context but deliberately degrade it:
   - Include only about half the key facts from the context.
   - Introduce at least one plausible but INCORRECT detail (a wrong number,
     wrong direction of effect, or wrong comparison).
   - The answer should still seem partially relevant but be demonstrably less
     faithful and less complete than the good answer.

3. BAD answer: Give an incorrect, vague, or completely off-topic answer.
   It should clearly contradict or ignore both the context and the ground truth.
   Do NOT just give a shorter version of the good answer.

CRITICAL JSON RULES:
- Respond ONLY with a valid JSON object.
- No markdown, no backticks, no extra text before or after.
- Do NOT use single quotes anywhere inside the JSON.
- Do NOT use apostrophes (') in the text — replace them with a space or rephrase.
- Do NOT use double quotes inside the answer text — rephrase instead.
- Use ONLY double quotes for JSON string delimiters.

{{"answer_good": "...", "answer_medium": "...", "answer_bad": "..."}}
"""



def generate_triplet(row: pd.Series, retries: int = 3) -> dict | None:
    context = "\n\n".join(row["retrieved_contexts"])
    prompt  = TRIPLET_PROMPT.format(
        context     = context[:3000],
        question    = row["question"],
        long_answer = row["ground_truth"],
    )
    for attempt in range(1, retries + 1):
        try:
            raw   = triplet_llm.invoke(prompt).content
            clean = re.sub(r"```(?:json)?", "", raw).strip().rstrip("`").strip()
            # Extract JSON object if there's extra text around it
            match = re.search(r'\{.*\}', clean, re.DOTALL)
            if match:
                clean = match.group(0)
            data  = json.loads(clean)
            # Normalize keys: "answer_good" → "good"
            return {k.replace("answer_", ""): v for k, v in data.items()}
        except Exception as e:
            print(f"  [TRIPLET WARNING] attempt {attempt}/{retries} q_id={row['question_id']}: {e}")
            if attempt == retries:
                return None
    return None


print("\nGenerating triplets...")
triplets = []
for _, row in tqdm(benchmark_questions.iterrows(), total=len(benchmark_questions)):
    t = generate_triplet(row)
    if t is None:
        continue
    for quality, answer in t.items():
        if quality not in {"good", "medium", "bad"}:
            continue  # skip unexpected keys
        triplets.append({
            "question_id"       : row["question_id"],
            "question"          : row["question"],
            "retrieved_contexts": row["retrieved_contexts"],
            "ground_truth"      : row["ground_truth"],
            "quality_level"     : quality,
            "expected_rank"     : {"good": 1, "medium": 2, "bad": 3}[quality],
            "generated_answer"  : answer,
        })

triplets_df = pd.DataFrame(triplets)
triplets_path = os.path.join(CONFIG_JUDGE["results_dir"], "benchmark_triplets.csv")
triplets_df.to_csv(triplets_path, index=False)
print(f"\n {len(triplets_df)} triplet rows generated and saved → {triplets_path}")
print(f"   Quality distribution:\n{triplets_df['quality_level'].value_counts().to_string()}")

In [ ]:
# PHASE 2 — 6: PROMPTS + CUSTOM METRICS
# Re-declare here so Phase 2B is self-contained.
# Identical to Phase 3 — all metrics are fully custom (no standard Ragas wrappers).
# CustomFaithfulness, CustomContextPrecision, CustomAnswerRelevancy all accept
# an explicit llm_client so the judge LLM can be swapped per judge candidate.

# ── SHARED HELPER ─────────────────────────────────────────────────────────────────
def parse_json_score(raw: str) -> float:
    if raw is None:
        return 0.0
    text = re.sub(r"```(?:json)?", "", str(raw)).strip().rstrip("`").strip()
    # Extrait uniquement le score par regex → reasoning ignoré automatiquement
    score_match = re.search(r'"score"\s*:\s*([0-9]*\.?[0-9]+)', text)
    if score_match:
        return max(0.0, min(1.0, float(score_match.group(1))))
    try:
        data = json.loads(re.search(r"\{.*\}", text, re.DOTALL).group(0))
        return max(0.0, min(1.0, float(data.get("score", 0.0))))
    except Exception as e:
        print(f"  [PARSE WARNING] {e} | raw[:150]: {raw[:150]}")
        return 0.0

# ── PROMPT + CLASS: FAITHFULNESS ───────────────────────────────────────────────
JUDGE_PROMPT_FAITHFULNESS = """\
You are an expert biomedical evaluator specialized in medical literature and RAG evaluation.

Your task is to evaluate FAITHFULNESS of the generated answer with respect to the retrieved contexts AND ANSWER CORRECTNESS FROM CONTEXT.

This metric checks two things:
1. Faithfulness: Are the factual claims in the generated answer supported by the retrieved contexts?
2. Correctness: Does the final answer or conclusion correctly follow from the retrieved contexts?

You must judge ONLY based on the retrieved contexts.
Do NOT use outside biomedical knowledge.
Do NOT reward an answer just because it sounds scientifically plausible.

=== Question ===
{question}

=== Retrieved Contexts ===
{contexts}

=== Generated Answer ===
{answer}

Evaluation rules:
1. Identify the main conclusion required by the question.
2. Identify the important factual claims made in the generated answer.
3. Check whether each important claim is supported or reasonably entailed by the retrieved contexts, without using outside biomedical knowledge.
4. For each important claim, decide whether it is:
   - fully supported by the retrieved contexts,
   - partially supported or reasonably entailed by the retrieved contexts,
   - unsupported by the retrieved contexts,
   - contradicted by the retrieved contexts.
5. For yes/no biomedical questions, determine whether the correct conclusion from the contexts is yes, no, mixed/nuanced, or insufficient evidence.
6. Compare the generated answer's conclusion with the conclusion supported by the contexts.
7. If the generated answer gives the opposite conclusion from the retrieved contexts, the score must be 0.4 or lower.
8. If the answer is relevant but the final conclusion is wrong, the score must be low.
9. If the retrieved contexts are insufficient and the answer correctly states that evidence is insufficient, give a high score.
10. If the retrieved contexts are insufficient but the answer invents a strong yes/no conclusion, give a low score.
11. Do not over-penalize minor wording differences if the factual meaning is supported.
12. If the answer contains both supported and unsupported claims, give a partial score proportional to the importance of the claims.
13. A concise answer with only supported claims should receive a high score.

Scoring instructions:
Assign a decimal score between 0 and 1 based on the degree of support and correctness.
Use any decimal value between 0 and 1 when appropriate.

The score should reflect:
- how many important claims in the generated answer are supported by the retrieved contexts,
- whether the final conclusion correctly follows from the retrieved contexts,
- whether the answer contains unsupported or contradicted information,
- whether the answer is overconfident when the contexts are insufficient,
- whether important biomedical nuances are missing.

Before scoring, think step by step:
- List the key claims in the generated answer.
- For each claim, check whether it is supported, partially supported, unsupported, or contradicted by the contexts.
- Determine whether the final conclusion is correct given the contexts.
- Based on this reasoning, decide the score.

Return ONLY valid compact JSON.
Do NOT use markdown.
Do NOT wrap the answer in ```json.
Do NOT use bullet points.
Do NOT use multiline explanations.
The reasoning and explanation must be short single-line strings.

Return exactly this format:
{{
  "reasoning": "<your step-by-step reasoning in one line>",
  "score": <float between 0 and 1>,
  "explanation": "<short single-line explanation>"
}}
"""

print("JUDGE_PROMPT_FAITHFULNESS defined.")

@dataclass
class CustomFaithfulness(SingleTurnMetric):
    name: str = "faithfulness"
    llm_client: object = field(default=None, repr=False)
    sleep_seconds: int = 3   # ← nouveau
    _required_columns: dict = field(default_factory=lambda: {
        MetricType.SINGLE_TURN: {"user_input", "response", "retrieved_contexts"}
    })

    def init(self, run_config=None):
        pass

    async def _single_turn_ascore(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:
        await asyncio.sleep(self.sleep_seconds)   # ← utilise le paramètre
        context = "\n\n".join(sample.retrieved_contexts or [])
        prompt  = JUDGE_PROMPT_FAITHFULNESS.format(
            question=sample.user_input, contexts=context, answer=sample.response
        )
        for attempt in range(1, 6):
            try:
                response = self.llm_client.invoke(prompt)
                raw      = response.content if hasattr(response, "content") else str(response)
                if not raw.strip():
                    print(f"  [Faithfulness] response vide — type: {type(response)}")
                    print(f"  [Faithfulness] response dict: {response.__dict__}")
                return parse_json_score(raw)
            except Exception as e:
                if attempt < 5:
                    wait = 120 * attempt   # ← 60s, 120s, 180s, 240s
                    print(f"  [Faithfulness] attempt {attempt}/5 failed: {e}. Waiting {wait}s...")
                    await asyncio.sleep(wait)
                else:
                    print(f"  [Faithfulness] all 5 attempts failed: {e}. Returning 0.0")
                    return 0.0

    def _single_turn_score(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:
        return asyncio.get_event_loop().run_until_complete(
            self._single_turn_ascore(sample, callbacks)
        )

print(" CustomFaithfulness (with swappable llm_client) ready.")

# ── PROMPT + CLASS: CONTEXT PRECISION ─────────────────────────────────────────
JUDGE_PROMPT_CONTEXT_PRECISION = """\
You are an expert biomedical evaluator specialized in RAG retrieval quality.

Your task is to evaluate CONTEXT PRECISION.

Context precision means:
Were the retrieved contexts that are useful for answering the question ranked early?

You must evaluate each retrieved context in order.

=== Question ===
{question}

=== Retrieved Contexts in Ranking Order ===
{contexts}

Evaluation rules:

For each context, decide whether it is useful for answering the exact question.

A context is useful if it contains evidence directly related to:
- the disease or biological condition in the question,
- the intervention, exposure, gene, protein, cell type, or model in the question,
- the outcome or mechanism asked in the question,
- or a direct result/conclusion needed to answer the question.

A context is NOT useful if:
- it is only general background,
- it mentions the broad topic but not the specific question,
- it is about a different disease, molecule, cell line, model, or mechanism,
- it is only methodological and does not help answer the question,
- it is irrelevant biomedical text from another study.

Use binary verdicts:
- 1 = useful for answering the question
- 0 = not useful for answering the question

Steps:
1. Assign verdict 1 or 0 to each context.
2. Compute precision at each rank where verdict = 1.
3. Average these precision values.
4. If there are no useful contexts, score = 0.

Before scoring, think step by step:
- For each chunk, state whether it is useful and why.
- Note which useful chunks appear early vs. late.
- Compute the precision@k mentally.
- Based on this reasoning, decide the score.

Return ONLY valid compact JSON.
Do NOT use markdown.
Do NOT wrap the answer in ```json.
Do NOT use bullet points.
Do NOT use multiline explanations.
The reasoning and explanation must be short single-line strings.

Return exactly this format:
{{
  "reasoning": "<your step-by-step reasoning in one line>",
  "score": <float between 0 and 1>,
  "verdicts": [0 or 1 for each context],
  "explanation": "<short single-line explanation>"
}}
"""

print("JUDGE_PROMPT_CONTEXT_PRECISION defined.")

@dataclass
class CustomContextPrecision(SingleTurnMetric):
    name: str = "context_precision"
    llm_client: object = field(default=None, repr=False)
    sleep_seconds: int = 3   # ← nouveau
    _required_columns: dict = field(default_factory=lambda: {
        MetricType.SINGLE_TURN: {
            "user_input", "retrieved_contexts", "reference"
        }
    })

    def init(self, run_config=None):
        pass

    async def _single_turn_ascore(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:
        await asyncio.sleep(self.sleep_seconds)   # ← utilise le paramètre
        chunks_text = "\n\n".join([
            f"Chunk {i+1}: {chunk}"
            for i, chunk in enumerate(sample.retrieved_contexts or [])
        ])
        prompt = JUDGE_PROMPT_CONTEXT_PRECISION.format(
            question     = sample.user_input,
            contexts     = chunks_text,
        )
        for attempt in range(1, 6):
            try:
                response = self.llm_client.invoke(prompt)
                raw = response.content if hasattr(response, "content") else str(response)
                if not raw.strip():
                    print(f"  [ContextPrecision] response vide — type: {type(response)}")
                    print(f"  [ContextPrecision] response dict: {response.__dict__}")
                return parse_json_score(raw)
            except Exception as e:
                if attempt < 5:
                    wait = 120 * attempt   # ← 60s, 120s, 180s, 240s
                    print(f"  [ContextPrecision] attempt {attempt}/5 failed: {e}. Waiting {wait}s...")
                    await asyncio.sleep(wait)
                else:
                    print(f"  [ContextPrecision] all 5 attempts failed: {e}. Returning 0.0")
                    return 0.0

    def _single_turn_score(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:
        return asyncio.get_event_loop().run_until_complete(
            self._single_turn_ascore(sample, callbacks)
        )

print(" CustomContextPrecision defined.")

# ── PROMPT + CLASS: ANSWER RELEVANCY ──────────────────────────────────────────
JUDGE_PROMPT_ANSWER_RELEVANCY = """\
You are an expert biomedical evaluator specialized in medical question answering.

Your task is to evaluate ANSWER RELEVANCY.

Answer relevancy means:
Does the generated answer directly address the question being asked?

Important:
This metric is NOT only about topic similarity.
The answer must respond to the specific biomedical question, including the correct entities, disease, intervention, mechanism, model, and outcome.

=== Question ===
{question}

=== Generated Answer ===
{answer}

Evaluation rules:
1. Check whether the answer directly answers the question.
2. For yes/no biomedical questions, the answer should clearly provide a yes/no or equivalent conclusion.
3. The answer must address the specific biomedical entities in the question.
4. Penalize answers that are generic, vague, off-topic, or only repeat background information.
5. Penalize answers that discuss the correct broad topic but miss the specific mechanism, disease, population, intervention, or outcome.
6. If the answer gives a conclusion that is clearly inconsistent with the biomedical question, the score must not exceed 0.6.
7. If the answer says the context is insufficient, this can still be relevant if it directly explains why the question cannot be answered.
8. Do NOT reward long answers automatically. A concise direct answer can receive a high score.

Scoring instructions:
Assign a decimal score between 0 and 1 based on how directly and specifically the answer addresses the question.
Use any decimal value between 0 and 1 when appropriate.

Before scoring, think step by step:
- Identify the specific biomedical entities, mechanism, and outcome required by the question.
- Check whether the answer addresses each of these specifically.
- Determine whether the answer is direct and specific or generic and vague.
- Based on this reasoning, decide the score.

Return ONLY valid compact JSON.
Do NOT use markdown.
Do NOT wrap the answer in ```json.
Do NOT use bullet points.
Do NOT use multiline explanations.
The reasoning and explanation must be short single-line strings.

Return exactly this format:
{{
  "reasoning": "<your step-by-step reasoning in one line>",
  "score": <float between 0 and 1>,
  "explanation": "<short single-line explanation>"
}}
"""

print("JUDGE_PROMPT_ANSWER_RELEVANCY defined.")

@dataclass
class CustomAnswerRelevancy(SingleTurnMetric):
    name: str = "answer_relevancy"
    llm_client: object = field(default=None, repr=False)
    sleep_seconds: int = 3   # ← nouveau
    _required_columns: dict = field(default_factory=lambda: {
        MetricType.SINGLE_TURN: {
            "user_input", "response", "reference"
        }
    })

    def init(self, run_config=None):
        pass

    async def _single_turn_ascore(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:
        await asyncio.sleep(self.sleep_seconds)   # ← utilise le paramètre
        prompt = JUDGE_PROMPT_ANSWER_RELEVANCY.format(
            question     = sample.user_input,
            answer       = sample.response,
        )
        for attempt in range(1, 6):
            try:
                response = self.llm_client.invoke(prompt)
                raw = response.content if hasattr(response, "content") else str(response)
                if not raw.strip():
                    print(f"  [AnswerRelevancy] response vide — type: {type(response)}")
                    print(f"  [AnswerRelevancy] response dict: {response.__dict__}")
                return parse_json_score(raw)
            except Exception as e:
                if attempt < 5:
                    wait = 120 * attempt   # ← 60s, 120s, 180s, 240s
                    print(f"  [AnswerRelevancy] attempt {attempt}/5 failed: {e}. Waiting {wait}s...")
                    await asyncio.sleep(wait)
                else:
                    print(f"  [AnswerRelevancy] all 5 attempts failed: {e}. Returning 0.0")
                    return 0.0

    def _single_turn_score(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:
        return asyncio.get_event_loop().run_until_complete(
            self._single_turn_ascore(sample, callbacks)
        )

print(" CustomAnswerRelevancy defined.")
print(" All prompts defined: FAITHFULNESS, CONTEXT_PRECISION, ANSWER_RELEVANCY")
print(" Custom metrics ready: CustomFaithfulness, CustomContextPrecision, CustomAnswerRelevancy")


In [ ]:
# PHASE 2 — 7: METRIC BUILDER
# Instantiates the exact same 4 metrics as Phase 3:
#   1. CustomFaithfulness       — binary claim-level faithfulness (custom prompt)
#   2. CustomContextPrecision   — Precision@k via custom LLM judge (custom prompt)
#   3. CustomAnswerRelevancy    — need-coverage score (custom prompt)
#   4. ContextRecall (Ragas)    — sentence-level recall with biomedical header
# All custom metrics accept an explicit llm_client for judge swapping.

def build_metrics_for_judge(judge_llm_raw, judge_llm_wrapped, judge_embeddings,  sleep_seconds=3):
    """
    Returns the list of 3 metrics configured for a specific judge LLM.
    Identical metric set to Phase 3 — fully custom except ContextRecall.

    Args:
        judge_llm_raw     : ChatNVIDIA instance  — used by all 3 custom metrics
        judge_llm_wrapped : LangchainLLMWrapper  — used by ContextRecall
        judge_embeddings  : LangchainEmbeddingsWrapper — kept for API compatibility
        sleep_seconds     : int — throttle between API calls (default 3s, use 10s for Mistral)
    Returns:
        list[SingleTurnMetric]
    """

    # 1. Custom Faithfulness — binary, 1 direct API call
    custom_faith = CustomFaithfulness(llm_client=judge_llm_raw, sleep_seconds=sleep_seconds)

    # 2. Custom Context Precision — Precision@k via custom LLM judge
    custom_cp = CustomContextPrecision(llm_client=judge_llm_raw, sleep_seconds=sleep_seconds)

    # 3. Custom Answer Relevancy — need-coverage score via custom LLM judge
    custom_ar = CustomAnswerRelevancy(llm_client=judge_llm_raw, sleep_seconds=sleep_seconds)

    # 4. Context Recall — standard Ragas with biomedical header
    

    return [custom_faith, custom_cp, custom_ar]


print("   build_metrics_for_judge() ready — using Phase 3 custom metrics.")
print("   Metrics: CustomFaithfulness | CustomContextPrecision | CustomAnswerRelevancy | ContextRecall")


In [ ]:
# PHASE 2 — 8: INITIALISE ALL 3 JUDGE LLMs + EMBEDDINGS

print("\nInitialising Judge A — Mistral-Large-675B...")
judge_A_raw     = ChatNVIDIA(
    model                 = CONFIG_JUDGE["judge_A_model"],
    temperature           = CONFIG_JUDGE["judge_temperature"],
    max_tokens            = CONFIG_JUDGE["judge_max_completion_tokens"],
)
judge_A_wrapped = LangchainLLMWrapper(judge_A_raw)
print(f"   Judge A ready: {CONFIG_JUDGE['judge_A_label']}")

print("\nInitialising Judge B — GPT-OSS-120B...")
judge_B_raw     = ChatNVIDIA(
    model                 = CONFIG_JUDGE["judge_B_model"],
    temperature           = CONFIG_JUDGE["judge_temperature"],
    max_tokens            = CONFIG_JUDGE["judge_max_completion_tokens"],
)
judge_B_wrapped = LangchainLLMWrapper(judge_B_raw)
print(f"   Judge B ready: {CONFIG_JUDGE['judge_B_label']}")

print("\nInitialising Judge C — Qwen3.5-397B...")
judge_C_raw     = ChatNVIDIA(
    model                 = CONFIG_JUDGE["judge_C_model"],
    temperature           = CONFIG_JUDGE["judge_temperature"],
    max_tokens            = CONFIG_JUDGE["judge_max_completion_tokens"],
)
judge_C_wrapped = LangchainLLMWrapper(judge_C_raw)
print(f"   Judge C ready: {CONFIG_JUDGE['judge_C_label']}")

print("\nLoading shared embedding model...")
hf_emb = NVIDIAEmbeddings(
    model   = CONFIG_JUDGE["embedding_model"],
    api_key = os.environ["NVIDIA_API_KEY"],
)
judge_embeddings = LangchainEmbeddingsWrapper(hf_emb)
print(f"   Embeddings ready: {CONFIG_JUDGE['embedding_model']}")

run_config_bench = RunConfig(
    timeout     = 1800,
    max_workers = 1,
    max_retries = 5,
)
print("\n RunConfig ready.")

In [ ]:
# PHASE 2 — 9: EVALUATE TRIPLETS WITH ONE JUDGE

def evaluate_triplets_with_judge(
    judge_label      : str,
    judge_llm_raw    : object,
    judge_llm_wrapped: object,
    judge_embeddings : object,
    triplets_df      : pd.DataFrame,
    run_config       : RunConfig,
    sleep_seconds    : int = 3,   # ← nouveau paramètre
) -> pd.DataFrame:

    print(f"\n{'='*60}")
    print(f"  Judge      : {judge_label}")
    print(f"  Samples    : {len(triplets_df)} triplet rows")
    print(f"{'='*60}")

    metrics = build_metrics_for_judge(
        judge_llm_raw     = judge_llm_raw,
        judge_llm_wrapped = judge_llm_wrapped,
        judge_embeddings  = judge_embeddings,
        sleep_seconds     = sleep_seconds,   # ← passé aux metrics
    )

    samples = [
        SingleTurnSample(
            user_input         = row["question"],
            response           = row["generated_answer"],
            retrieved_contexts = row["retrieved_contexts"],
            reference          = row["ground_truth"],
        )
        for _, row in triplets_df.iterrows()
    ]

    dataset = EvaluationDataset(samples=samples)
    result  = evaluate(dataset=dataset, metrics=metrics, run_config=run_config)
    scores  = result.to_pandas()

    scores.insert(0, "judge_label",   judge_label)
    scores.insert(1, "question_id",   triplets_df["question_id"].values)
    scores.insert(2, "quality_level", triplets_df["quality_level"].values)
    scores.insert(3, "expected_rank", triplets_df["expected_rank"].values)
    scores = scores.rename(columns={
        "user_input": "question",
        "response"  : "generated_answer",
        "reference" : "ground_truth",
    })

    scores["overall_score"] = scores[
        ["faithfulness", "answer_relevancy", "context_precision"]
    ].mean(axis=1)

    print(f"   Done — {judge_label}")
    for m in ["faithfulness", "answer_relevancy", "context_precision"]:
        print(f"     {m:<22}: {scores[m].mean():.3f}")
    return scores


print(" evaluate_triplets_with_judge() ready.")

In [ ]:
import os
import ast
import pandas as pd
from tqdm import tqdm

triplets_path = os.path.join(CONFIG_JUDGE["results_dir"], "benchmark_triplets.csv")

if os.path.exists(triplets_path):
    print(f" Loading existing triplets from: {triplets_path}")
    triplets_df = pd.read_csv(triplets_path)

    # convert stringified lists back to Python lists if needed
    if "retrieved_contexts" in triplets_df.columns:
        triplets_df["retrieved_contexts"] = triplets_df["retrieved_contexts"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )

    print(f" triplets_df loaded — {len(triplets_df)} rows")

else:
    print("No saved triplets found. Generating new triplets...")

    triplets = []

    for _, row in tqdm(benchmark_questions.iterrows(), total=len(benchmark_questions)):
        t = generate_triplet(row)   # this must return 3 answers: good / medium / bad

        if t is None:
            continue

        for quality, answer in t.items():
            # supports keys like answer_good / answer_medium / answer_bad
            clean_quality = quality.replace("answer_", "")

            triplets.append({
                "question_id": row["question_id"],
                "question": row["question"],
                "retrieved_contexts": row["retrieved_contexts"],
                "ground_truth": row["ground_truth"],
                "quality_level": clean_quality,
                "expected_rank": {
                    "good": 1,
                    "medium": 2,
                    "bad": 3
                }[clean_quality],
                "generated_answer": answer,
            })

    triplets_df = pd.DataFrame(triplets)
    triplets_df.to_csv(triplets_path, index=False)

    print(f" triplets generated — {len(triplets_df)} rows")
    print(f" Saved to: {triplets_path}")

display(triplets_df.head())

In [ ]:
# PHASE 2 — 10: RUN JUDGE A — Mistral-Large-675B

print("\n" + "="*60)
print(f"  Running Judge A: {CONFIG_JUDGE['judge_A_label']}")
print("="*60)

scores_A = evaluate_triplets_with_judge(
    judge_label       = CONFIG_JUDGE["judge_A_label"],
    judge_llm_raw     = judge_A_raw,
    judge_llm_wrapped = judge_A_wrapped,
    judge_embeddings  = judge_embeddings,
    triplets_df       = triplets_df,
    run_config        = run_config_bench,
    sleep_seconds     = CONFIG_JUDGE["judge_A_sleep_seconds"],   # ← 10s pour Mistral
)

path_A = os.path.join(
    CONFIG_JUDGE["results_dir"],
    f"judge_scores_{CONFIG_JUDGE['judge_A_label'].replace(' ','_')}.csv"
)
scores_A.to_csv(path_A, index=False)
print(f"\n Judge A scores saved → {path_A}")

print("\nWaiting 60s before Judge B...")
time.sleep(60)

In [ ]:
path_A = os.path.join(
    CONFIG_JUDGE["results_dir"],
    f"judge_scores_{CONFIG_JUDGE['judge_A_label'].replace(' ','_')}.csv"
)
scores_A.to_csv(path_A, index=False)
print(f"\n Judge A scores saved → {path_A}")

print("\nWaiting 60s before Judge B...")
time.sleep(60)

In [ ]:
# PHASE 2 — 11: RUN JUDGE B — GPT-OSS-120B

print("\n" + "="*60)
print(f"  Running Judge B: {CONFIG_JUDGE['judge_B_label']}")
print("="*60)

scores_B = evaluate_triplets_with_judge(
    judge_label      = CONFIG_JUDGE["judge_B_label"],
    judge_llm_raw    = judge_B_raw,
    judge_llm_wrapped= judge_B_wrapped,
    judge_embeddings = judge_embeddings,
    triplets_df      = triplets_df,
    run_config       = run_config_bench,
)

path_B = os.path.join(
    CONFIG_JUDGE["results_dir"],
    f"judge_scores_{CONFIG_JUDGE['judge_B_label'].replace(' ','_')}.csv"
)
scores_B.to_csv(path_B, index=False)
print(f"\n Judge B scores saved → {path_B}")

print("\nWaiting 60s before Judge C...")
time.sleep(60)

In [ ]:
# PHASE 2 — 11b: RUN JUDGE C — Qwen3.5-397B

print("\n" + "="*60)
print(f"  Running Judge C: {CONFIG_JUDGE['judge_C_label']}")
print("="*60)

scores_C = evaluate_triplets_with_judge(
    judge_label      = CONFIG_JUDGE["judge_C_label"],
    judge_llm_raw    = judge_C_raw,
    judge_llm_wrapped= judge_C_wrapped,
    judge_embeddings = judge_embeddings,
    triplets_df      = triplets_df,
    run_config       = run_config_bench,
)

path_C = os.path.join(
    CONFIG_JUDGE["results_dir"],
    f"judge_scores_{CONFIG_JUDGE['judge_C_label'].replace(' ','_')}.csv"
)
scores_C.to_csv(path_C, index=False)
print(f"\n Judge C scores saved → {path_C}")

In [ ]:
# PHASE 2 — 12: BENCHMARK — 3 JUDGES COMPARISON

SCORE_COLS    = ["faithfulness","answer_relevancy",
                 "context_precision"]
QUALITY_ORDER = {"good": 1, "medium": 2, "bad": 3}


def compute_ranking_accuracy(scores_df: pd.DataFrame) -> float:
    correct = 0
    total   = 0
    for qid, grp in scores_df.groupby("question_id"):
        if len(grp) < 3:
            continue
        grp = grp.set_index("quality_level")
        if not all(q in grp.index for q in ["good","medium","bad"]):
            continue
        s_good   = grp.loc["good",   "overall_score"]
        s_medium = grp.loc["medium", "overall_score"]
        s_bad    = grp.loc["bad",    "overall_score"]
        if s_good > s_medium > s_bad:
            correct += 1
        total += 1
    return correct / total if total > 0 else 0.0


def compute_pearson(scores_df: pd.DataFrame) -> float:
    merged = scores_df[["expected_rank","overall_score"]].dropna()
    r, _   = pearsonr(-merged["expected_rank"], merged["overall_score"])
    return r


def compute_score_variance(scores_df: pd.DataFrame) -> float:
    variances = []
    for qid, grp in scores_df.groupby("question_id"):
        if len(grp) >= 2:
            variances.append(grp["overall_score"].var())
    return np.mean(variances) if variances else 0.0


# Add expected_rank and overall_score to each judge results
for df in [scores_B, scores_C]:
    df["expected_rank"] = df["quality_level"].map(QUALITY_ORDER)
    df["overall_score"] = df[SCORE_COLS].mean(axis=1)

# Compute metrics for all 3 judges
comparison_rows = []
for label, df in [
    (CONFIG_JUDGE["judge_B_label"], scores_B),
    (CONFIG_JUDGE["judge_C_label"], scores_C),
]:
    comparison_rows.append({
        "judge_label"      : label,
        "ranking_accuracy" : compute_ranking_accuracy(df),
        "pearson_r"        : compute_pearson(df),
        "mean_variance"    : compute_score_variance(df),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(
    "ranking_accuracy", ascending=False
)

# Save comparison
comp_path = os.path.join(CONFIG_JUDGE["results_dir"], "judge_comparison.csv")
comparison_df.to_csv(comp_path, index=False)

print("\n" + "="*60)
print("  JUDGE COMPARISON RESULTS")
print("="*60)
print(comparison_df.to_string(index=False))

In [ ]:
# PHASE 2 — 13: VISUALISATION — 3 JUDGES

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle(
    "Phase 2B — Judge Comparison (3 candidates)",
    fontsize=14, fontweight="bold"
)

metrics_to_plot = [
    ("ranking_accuracy", "Ranking Accuracy\n(perfect good>med>bad)", "steelblue"),
    ("pearson_r",        "Pearson Correlation\n(expected vs actual)", "darkorange"),
    ("mean_variance",    "Score Variance\n(discrimination power)",    "seagreen"),
]

labels = comparison_df["judge_label"].tolist()
colors = ["steelblue", "darkorange", "seagreen"]

for ax, (col, title, color) in zip(axes, metrics_to_plot):
    vals = comparison_df[col].tolist()
    bars = ax.bar(labels, vals, color=colors, alpha=0.8, edgecolor="black")
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, max(vals) * 1.3 + 0.05)
    ax.set_ylabel("Score")
    ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=8)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{v:.3f}",
            ha="center", va="bottom",
            fontsize=9, fontweight="bold",
        )

plt.tight_layout()
plot_path = os.path.join(
    CONFIG_JUDGE["results_dir"], "judge_comparison_3.png"
)
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n Chart saved → {plot_path}")

In [ ]:
# PHASE 2 — 13: VISUALISATION — 3 JUDGES (toutes les visualisations)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

labels = comparison_df["judge_label"].tolist()
colors = ["#4472C4", "#ED7D31", "#4CAF50"]

ranking_acc  = comparison_df["ranking_accuracy"].tolist()
pearson_corr = comparison_df["pearson_r"].tolist()
score_var    = comparison_df["mean_variance"].tolist()

data = np.array([ranking_acc, pearson_corr, score_var])  # (3 metrics x 3 models)
metrics_labels = ["Ranking Accuracy", "Pearson Correlation", "Score Variance"]

# ──────────────────────────────────────────────────────────────────────────────
# VIZ 1 : Graphe original (barres verticales)
# ──────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle("Phase 2B — Judge Comparison (3 candidates)", fontsize=14, fontweight="bold")

metrics_to_plot = [
    ("ranking_accuracy", "Ranking Accuracy\n(perfect good>med>bad)"),
    ("pearson_r",        "Pearson Correlation\n(expected vs actual)"),
    ("mean_variance",    "Score Variance\n(discrimination power)"),
]

for ax, (col, title) in zip(axes, metrics_to_plot):
    vals = comparison_df[col].tolist()
    bars = ax.bar(labels, vals, color=colors, alpha=0.85, edgecolor="white", linewidth=0.8)
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, max(vals) * 1.3 + 0.05)
    ax.set_ylabel("Score")
    ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_facecolor("#f9f9f9")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    for bar, v, i in zip(bars, vals, range(3)):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9,
                fontweight="bold" if i == 0 else "normal")

plt.tight_layout()
plot_path = os.path.join(CONFIG_JUDGE["results_dir"], "judge_comparison_3.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ VIZ 1 — Barres verticales → {plot_path}")

# ──────────────────────────────────────────────────────────────────────────────
# VIZ 2 : Heatmap
# ──────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor("white")

norm = data / data.max(axis=1, keepdims=True)
im = ax.imshow(norm.T, cmap="Blues", aspect="auto", vmin=0.5, vmax=1.0)

ax.set_xticks(range(3))
ax.set_xticklabels(metrics_labels, fontsize=11)
ax.set_yticks(range(3))
ax.set_yticklabels(labels, fontsize=11)

for i in range(3):
    for j in range(3):
        val_raw  = data[i, j]
        val_norm = norm[i, j]
        txt_color = "white" if val_norm > 0.85 else "#222"
        ax.text(i, j, f"{val_raw:.3f}", ha="center", va="center",
                fontsize=13, fontweight="bold" if j == 0 else "normal",
                color=txt_color)

cb = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
cb.set_label("Normalized score", fontsize=9)
ax.set_title("Phase 2B — Performance Heatmap", fontsize=13, fontweight="bold", pad=12)

plt.tight_layout()
heatmap_path = os.path.join(CONFIG_JUDGE["results_dir"], "judge_comparison_heatmap.png")
plt.savefig(heatmap_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✅ VIZ 2 — Heatmap → {heatmap_path}")

# ──────────────────────────────────────────────────────────────────────────────
# VIZ 3 : Bubble Chart (Ranking Accuracy vs Pearson, taille = Score Variance)
# ──────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor("white")
ax.set_facecolor("#f9f9f9")

sizes = [v * 8000 for v in score_var]
ax.scatter(ranking_acc, pearson_corr, s=sizes, c=colors, alpha=0.75,
           edgecolors="white", linewidths=2, zorder=3)

xlim = 1.0
for i, (x, y, name, sv) in enumerate(zip(ranking_acc, pearson_corr, labels, score_var)):
    ax.annotate(f"{name}\n(var={sv:.3f})", (x, y),
                xytext=(x, y + 0.018),
                fontsize=9, ha="center",
                fontweight="bold" if i == 0 else "normal",
                color=colors[i])

ax.set_xlabel("Ranking Accuracy", fontsize=11)
ax.set_ylabel("Pearson Correlation", fontsize=11)
ax.set_title("Phase 2B — Bubble Chart\n(bubble size = Score Variance)",
             fontsize=12, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(linestyle="--", alpha=0.4)
legend_patches = [mpatches.Patch(color=c, label=m) for c, m in zip(colors, labels)]
ax.legend(handles=legend_patches, fontsize=9, loc="lower right")

plt.tight_layout()
bubble_path = os.path.join(CONFIG_JUDGE["results_dir"], "judge_comparison_bubble.png")
plt.savefig(bubble_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✅ VIZ 3 — Bubble Chart → {bubble_path}")

# ──────────────────────────────────────────────────────────────────────────────
# VIZ 4 : Score Cards + Barres normalisées
# ──────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 7))
fig.patch.set_facecolor("white")
gs = gridspec.GridSpec(2, 3, height_ratios=[1, 3], hspace=0.45, wspace=0.35)

best_vals  = [ranking_acc[0], pearson_corr[0], score_var[0]]
second_best = [max(ranking_acc[1], ranking_acc[2]),
               max(pearson_corr[1], pearson_corr[2]),
               max(score_var[1], score_var[2])]
card_titles = ["Ranking Accuracy", "Pearson Correlation", "Score Variance"]
card_deltas = [f"+{(best_vals[i]-second_best[i])*100:.1f}pp" for i in range(3)]

for j in range(3):
    ax = fig.add_subplot(gs[0, j])
    ax.set_facecolor("#EBF0FA")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis("off")
    ax.text(0.5, 0.72, card_titles[j], ha="center", fontsize=10,
            fontweight="bold", color="#2a52a0")
    ax.text(0.5, 0.38, f"{best_vals[j]:.3f}", ha="center", fontsize=20,
            fontweight="bold", color="#4472C4")
    ax.text(0.5, 0.08, f"Mistral leads {card_deltas[j]}", ha="center",
            fontsize=8.5, color="#555")

ax_main = fig.add_subplot(gs[1, :])
norm2 = data / data.max(axis=1, keepdims=True)
x = np.arange(3)
w = 0.25

for i, (model, color) in enumerate(zip(labels, colors)):
    offset = (i - 1) * w
    bars = ax_main.bar(x + offset, norm2[:, i], w, color=color,
                       label=model, edgecolor="white", linewidth=0.8)
    for bar, raw in zip(bars, data[:, i]):
        ax_main.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.01, f"{raw:.3f}",
                     ha="center", fontsize=8,
                     fontweight="bold" if i == 0 else "normal")

ax_main.set_xticks(x)
ax_main.set_xticklabels(metrics_labels, fontsize=11)
ax_main.set_ylabel("Normalized Score (0–1)", fontsize=10)
ax_main.set_ylim(0, 1.18)
ax_main.set_facecolor("#f9f9f9")
ax_main.spines["top"].set_visible(False)
ax_main.spines["right"].set_visible(False)
ax_main.grid(axis="y", linestyle="--", alpha=0.4)
ax_main.legend(fontsize=10, loc="upper right", framealpha=0.9)
ax_main.set_title("Phase 2B — Normalized Performance (all metrics on same scale)",
                  fontsize=12, fontweight="bold")

plt.savefig(os.path.join(CONFIG_JUDGE["results_dir"], "judge_comparison_scorecards.png"),
            dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✅ VIZ 4 — Score Cards + Barres normalisées")

# ──────────────────────────────────────────────────────────────────────────────
# VIZ 5 : Bump Chart (classements)
# ──────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor("white")
ax.set_facecolor("#f9f9f9")

ranks = []
for mi in range(3):
    order = np.argsort(data[mi])[::-1]
    r = np.empty_like(order)
    r[order] = np.arange(1, 4)
    ranks.append(r)
ranks = np.array(ranks)

x = [0, 1, 2]
for i, (model, color) in enumerate(zip(labels, colors)):
    y = ranks[:, i]
    ax.plot(x, y, color=color, linewidth=3, marker="o", markersize=12,
            markeredgecolor="white", markeredgewidth=2, zorder=3, label=model)
    for xi, yi in zip(x, y):
        ax.text(xi, yi - 0.18, f"#{yi}", ha="center", va="top",
                fontsize=10, fontweight="bold", color=color)

ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, fontsize=11)
ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["1st", "2nd", "3rd"], fontsize=11)
ax.invert_yaxis()
ax.set_xlim(-0.3, 2.3)
ax.set_ylim(3.5, 0.5)
ax.set_title("Phase 2B — Ranking Bump Chart\n(position across metrics)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10, loc="lower right", framealpha=0.9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
bump_path = os.path.join(CONFIG_JUDGE["results_dir"], "judge_comparison_bump.png")
plt.savefig(bump_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✅ VIZ 5 — Bump Chart → {bump_path}")

In [ ]:
# PHASE 2 — 14: SELECT WINNER FROM 3 JUDGES

winner_row = comparison_df.sort_values(
    ["ranking_accuracy", "pearson_r"], ascending=False
).iloc[0]

SELECTED_JUDGE_LABEL = winner_row["judge_label"]

# Map winner label to its objects
judge_map = {
    CONFIG_JUDGE["judge_A_label"]: (
        CONFIG_JUDGE["judge_A_model"], judge_A_raw, judge_A_wrapped
    ),
    CONFIG_JUDGE["judge_B_label"]: (
        CONFIG_JUDGE["judge_B_model"], judge_B_raw, judge_B_wrapped
    ),
    CONFIG_JUDGE["judge_C_label"]: (
        CONFIG_JUDGE["judge_C_model"], judge_C_raw, judge_C_wrapped
    ),
}

SELECTED_JUDGE_MODEL   = judge_map[SELECTED_JUDGE_LABEL][0]
SELECTED_JUDGE_RAW     = judge_map[SELECTED_JUDGE_LABEL][1]
SELECTED_JUDGE_WRAPPED = judge_map[SELECTED_JUDGE_LABEL][2]

print("\n" + "="*60)
print(f"  ✅ WINNER: {SELECTED_JUDGE_LABEL}")
print(f"     Model  : {SELECTED_JUDGE_MODEL}")
print(f"     ranking_accuracy : {winner_row['ranking_accuracy']:.4f}")
print(f"     pearson_r        : {winner_row['pearson_r']:.4f}")
print(f"     mean_variance    : {winner_row['mean_variance']:.4f}")
print("="*60)

print("\n  Full ranking:")
for _, row in comparison_df.iterrows():
    medal = "🥇" if row["judge_label"] == SELECTED_JUDGE_LABEL else "  "
    print(f"  {medal} {row['judge_label']:30s} "
          f"rank_acc={row['ranking_accuracy']:.3f}  "
          f"pearson={row['pearson_r']:.3f}")

print(f"\n  → SELECTED_JUDGE_RAW, SELECTED_JUDGE_WRAPPED")
print(f"    are ready for Phase 3 evaluation.")

In [ ]:
# PHASE 2 — 12: STANDARD RAGAS METRICS COMPARISON
# Using the best judge selected (Mistral-Large-675B) to evaluate the same triplets
# with standard Ragas metrics (Faithfulness, AnswerRelevancy, ContextPrecision)
# for comparison with our custom metrics.

from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample

print("\n" + "="*60)
print(f"  Standard Ragas Metrics — Judge: {CONFIG_JUDGE['judge_A_label']}")
print("="*60)

# ── Build standard metrics with Mistral as LLM backend ────────────────────────
std_faithfulness       = Faithfulness(llm=judge_A_wrapped)
std_answer_relevancy   = AnswerRelevancy(llm=judge_A_wrapped, embeddings=judge_embeddings)
std_context_precision  = ContextPrecision(llm=judge_A_wrapped)

std_metrics = [std_faithfulness, std_answer_relevancy, std_context_precision]

# ── Build samples (same triplets as custom evaluation) ────────────────────────
std_samples = [
    SingleTurnSample(
        user_input         = row["question"],
        response           = row["generated_answer"],
        retrieved_contexts = row["retrieved_contexts"],
        reference          = row["ground_truth"],
    )
    for _, row in triplets_df.iterrows()
]

std_dataset = EvaluationDataset(samples=std_samples)

# ── Evaluate ──────────────────────────────────────────────────────────────────
print("\nRunning standard Ragas evaluation...")
std_result = evaluate(
    dataset    = std_dataset,
    metrics    = std_metrics,
    run_config = run_config_bench,
)
std_scores = std_result.to_pandas()

# ── Add metadata columns ──────────────────────────────────────────────────────
std_scores.insert(0, "judge_label",   CONFIG_JUDGE["judge_A_label"])
std_scores.insert(1, "question_id",   triplets_df["question_id"].values)
std_scores.insert(2, "quality_level", triplets_df["quality_level"].values)
std_scores.insert(3, "expected_rank", triplets_df["expected_rank"].values)
std_scores = std_scores.rename(columns={
    "user_input": "question",
    "response"  : "generated_answer",
    "reference" : "ground_truth",
})
std_scores["overall_score"] = std_scores[
    ["faithfulness", "answer_relevancy", "context_precision"]
].mean(axis=1)

# ── Save ──────────────────────────────────────────────────────────────────────
path_std = os.path.join(CONFIG_JUDGE["results_dir"], "judge_scores_standard_ragas.csv")
std_scores.to_csv(path_std, index=False)
print(f"\n✅ Standard Ragas scores saved → {path_std}")

# ── Print scores by quality level ─────────────────────────────────────────────
print("\n=== Standard Ragas — scores by quality_level ===")
print(std_scores.groupby("quality_level")[
    ["faithfulness","answer_relevancy","context_precision","overall_score"]
].mean().round(3))

# ── Comparison: Custom vs Standard ────────────────────────────────────────────
print("\n" + "="*60)
print("  COMPARISON — Custom Prompts vs Standard Ragas")
print("  Judge: Mistral-Large-675B | Same triplets")
print("="*60)

# Load custom scores for Mistral
custom_scores = scores_A.copy()

comparison = pd.DataFrame({
    "metric"         : ["faithfulness", "answer_relevancy", "context_precision", "overall_score"],
    "custom_good"    : [custom_scores[custom_scores["quality_level"]=="good"][m].mean()
                        for m in ["faithfulness","answer_relevancy","context_precision","overall_score"]],
    "standard_good"  : [std_scores[std_scores["quality_level"]=="good"][m].mean()
                        for m in ["faithfulness","answer_relevancy","context_precision","overall_score"]],
    "custom_medium"  : [custom_scores[custom_scores["quality_level"]=="medium"][m].mean()
                        for m in ["faithfulness","answer_relevancy","context_precision","overall_score"]],
    "standard_medium": [std_scores[std_scores["quality_level"]=="medium"][m].mean()
                        for m in ["faithfulness","answer_relevancy","context_precision","overall_score"]],
    "custom_bad"     : [custom_scores[custom_scores["quality_level"]=="bad"][m].mean()
                        for m in ["faithfulness","answer_relevancy","context_precision","overall_score"]],
    "standard_bad"   : [std_scores[std_scores["quality_level"]=="bad"][m].mean()
                        for m in ["faithfulness","answer_relevancy","context_precision","overall_score"]],
}).round(3)

print(comparison.to_string(index=False))

# ── Ranking accuracy comparison ───────────────────────────────────────────────
def ranking_accuracy(df):
    correct = 0
    total   = 0
    for qid in df["question_id"].unique():
        q = df[df["question_id"] == qid].set_index("quality_level")
        if set(["good","medium","bad"]).issubset(q.index):
            if q.loc["good","overall_score"] > q.loc["medium","overall_score"] > q.loc["bad","overall_score"]:
                correct += 1
            total += 1
    return round(correct / total, 3) if total > 0 else 0.0

print(f"\n  Ranking Accuracy — Custom  : {ranking_accuracy(custom_scores)}")
print(f"  Ranking Accuracy — Standard: {ranking_accuracy(std_scores)}")
print("\n✅ Comparison done.")

---

### **PHASE 3 : LLM-as-a-judge**

Judge  : mistralai/mistral-large-3-675b-instruct-2512 (NVIDIA Build API) <br>
Input  : results/combined_rag_results.csv (from Phase 1) <br>
Output : results/ragas_judge_results.csv <br>
         results/ragas_judge_summary.csv <br>

In [ ]:
!pip install -q \
  nest_asyncio \
  ragas>=0.2.0 \
  langchain-nvidia-ai-endpoints \
  langchain-huggingface \
  sentence-transformers \
  pandas \
  tqdm

In [ ]:
import nest_asyncio
nest_asyncio.apply()
print(" nest_asyncio applied.")

In [ ]:
import os
import re
import json
import asyncio
import getpass
import time
import pandas as pd
from tqdm import tqdm
from dataclasses import dataclass, field

# Ragas
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics.base import SingleTurnMetric, MetricType
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

# LangChain + NVIDIA
from langchain_nvidia_ai_endpoints import (
    ChatNVIDIA,
    NVIDIAEmbeddings,
)

print(" All imports successful.")


In [ ]:
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass(
        "Enter your NVIDIA API key: "
    )

if os.environ["NVIDIA_API_KEY"].startswith("nvapi-"):
    print(" NVIDIA API key loaded.")
else:
    print("  Warning: key may be invalid.")

In [ ]:
CONFIG = {
    # Input from Phase 1
    "combined_results_path" : os.path.join(
        "results", "RAGOutput", "combined_rag_results.csv"
    ),

    "judge_model"                : "openai/gpt-oss-120b",
    "judge_temperature"          : 0.0,
    "judge_max_completion_tokens": 2048,


    # Embedding model — same as Phase 1 for consistency
    "embedding_model"            : "nvidia/llama-nemotron-embed-1b-v2",

    # Batch settings — prevents 429 Too Many Requests
    "batch_size"                 : 5,    # samples per batch
    "sleep_between_batches"      : 90,   # seconds between batches
    "sleep_between_models"       : 180,  # seconds between models

    # Output
    "results_dir"                : "results",
    "ragas_output_dir" : os.path.join("results", "RAGASOutput"),
}

os.makedirs(CONFIG["results_dir"], exist_ok=True)
os.makedirs(CONFIG["ragas_output_dir"], exist_ok=True)
print(" Configuration ready.")
print(f"   Judge       : {CONFIG['judge_model']}")
print(f"   Batch size  : {CONFIG['batch_size']} samples")
print(f"   Sleep       : {CONFIG['sleep_between_batches']}s between batches")


In [ ]:
print("\nLoading Phase 1 results...")
combined_df = pd.read_csv(CONFIG["combined_results_path"])

combined_df["retrieved_contexts"] = (
    combined_df["retrieved_contexts_json"].apply(json.loads)
)

print(f" Loaded: {len(combined_df)} rows")
print(f"   Models  : {combined_df['model_key'].unique().tolist()}")
print(f"   Columns : {list(combined_df.columns)}")

In [ ]:
print("\nInitializing Judge LLM...")
nvidia_llm = ChatNVIDIA(
    model                 = CONFIG["judge_model"],
    temperature           = CONFIG["judge_temperature"],
    max_completion_tokens = CONFIG["judge_max_completion_tokens"],
)
judge_llm = LangchainLLMWrapper(nvidia_llm)

#test = nvidia_llm.invoke("Reply with the single word: OK")
print(f" Judge LLM ready: {CONFIG['judge_model']}")
#print(f"   Test response : {test.content.strip()}")

In [ ]:
print("\nInitializing NVIDIA embedding model...")
judge_embeddings = LangchainEmbeddingsWrapper(
    NVIDIAEmbeddings(
        model   = CONFIG["embedding_model"],
        api_key = os.environ["NVIDIA_API_KEY"],
    )
)
print(f" Embeddings ready: {CONFIG['embedding_model']}")
print(f"   Same model as Phase 1 RAG ")

In [ ]:
JUDGE_PROMPT_FAITHFULNESS = """\
You are an expert biomedical evaluator specialized in medical literature and RAG evaluation.

Your task is to evaluate FAITHFULNESS of the generated answer with respect to the retrieved contexts AND ANSWER CORRECTNESS FROM CONTEXT.

This metric checks two things:
1. Faithfulness: Are the factual claims in the generated answer supported by the retrieved contexts?
2. Correctness: Does the final answer or conclusion correctly follow from the retrieved contexts?

You must judge ONLY based on the retrieved contexts.
Do NOT use outside biomedical knowledge.
Do NOT reward an answer just because it sounds scientifically plausible.

=== Question ===
{question}

=== Retrieved Contexts ===
{contexts}

=== Generated Answer ===
{answer}

Evaluation rules:
1. Identify the main conclusion required by the question.
2. Identify the important factual claims made in the generated answer.
3. Check whether each important claim is supported or reasonably entailed by the retrieved contexts, without using outside biomedical knowledge.
4. For each important claim, decide whether it is:
   - fully supported by the retrieved contexts,
   - partially supported or reasonably entailed by the retrieved contexts,
   - unsupported by the retrieved contexts,
   - contradicted by the retrieved contexts.
5. For yes/no biomedical questions, determine whether the correct conclusion from the contexts is yes, no, mixed/nuanced, or insufficient evidence.
6. Compare the generated answer's conclusion with the conclusion supported by the contexts.
7. If the generated answer gives the opposite conclusion from the retrieved contexts, the score must be 0.4 or lower.
8. If the answer is relevant but the final conclusion is wrong, the score must be low.
9. If the retrieved contexts are insufficient and the answer correctly states that evidence is insufficient, give a high score.
10. If the retrieved contexts are insufficient but the answer invents a strong yes/no conclusion, give a low score.
11. Do not over-penalize minor wording differences if the factual meaning is supported.
12. If the answer contains both supported and unsupported claims, give a partial score proportional to the importance of the claims.
13. A concise answer with only supported claims should receive a high score.

Scoring instructions:
Assign a decimal score between 0 and 1 based on the degree of support and correctness.
Use any decimal value between 0 and 1 when appropriate.

The score should reflect:
- how many important claims in the generated answer are supported by the retrieved contexts,
- whether the final conclusion correctly follows from the retrieved contexts,
- whether the answer contains unsupported or contradicted information,
- whether the answer is overconfident when the contexts are insufficient,
- whether important biomedical nuances are missing.

Before scoring, think step by step:
- List the key claims in the generated answer.
- For each claim, check whether it is supported, partially supported, unsupported, or contradicted by the contexts.
- Determine whether the final conclusion is correct given the contexts.
- Based on this reasoning, decide the score.

Return ONLY valid compact JSON.
Do NOT use markdown.
Do NOT wrap the answer in ```json.
Do NOT use bullet points.
Do NOT use multiline explanations.
The reasoning and explanation must be short single-line strings.

Return exactly this format:
{{
  "reasoning": "<your step-by-step reasoning in one line>",
  "score": <float between 0 and 1>,
  "explanation": "<short single-line explanation>"
}}
"""

print("JUDGE_PROMPT_FAITHFULNESS defined.")

In [ ]:
JUDGE_PROMPT_ANSWER_RELEVANCY = """\
You are an expert biomedical evaluator specialized in medical question answering.

Your task is to evaluate ANSWER RELEVANCY.

Answer relevancy means:
Does the generated answer directly address the question being asked?

Important:
This metric is NOT only about topic similarity.
The answer must respond to the specific biomedical question, including the correct entities, disease, intervention, mechanism, model, and outcome.

=== Question ===
{question}

=== Generated Answer ===
{answer}

Evaluation rules:
1. Check whether the answer directly answers the question.
2. For yes/no biomedical questions, the answer should clearly provide a yes/no or equivalent conclusion.
3. The answer must address the specific biomedical entities in the question.
4. Penalize answers that are generic, vague, off-topic, or only repeat background information.
5. Penalize answers that discuss the correct broad topic but miss the specific mechanism, disease, population, intervention, or outcome.
6. If the answer gives a conclusion that is clearly inconsistent with the biomedical question, the score must not exceed 0.6.
7. If the answer says the context is insufficient, this can still be relevant if it directly explains why the question cannot be answered.
8. Do NOT reward long answers automatically. A concise direct answer can receive a high score.

Scoring instructions:
Assign a decimal score between 0 and 1 based on how directly and specifically the answer addresses the question.
Use any decimal value between 0 and 1 when appropriate.

Before scoring, think step by step:
- Identify the specific biomedical entities, mechanism, and outcome required by the question.
- Check whether the answer addresses each of these specifically.
- Determine whether the answer is direct and specific or generic and vague.
- Based on this reasoning, decide the score.

Return ONLY valid compact JSON.
Do NOT use markdown.
Do NOT wrap the answer in ```json.
Do NOT use bullet points.
Do NOT use multiline explanations.
The reasoning and explanation must be short single-line strings.

Return exactly this format:
{{
  "reasoning": "<your step-by-step reasoning in one line>",
  "score": <float between 0 and 1>,
  "explanation": "<short single-line explanation>"
}}
"""

print("JUDGE_PROMPT_ANSWER_RELEVANCY defined.")

In [ ]:
JUDGE_PROMPT_CONTEXT_PRECISION = """\
You are an expert biomedical evaluator specialized in RAG retrieval quality.

Your task is to evaluate CONTEXT PRECISION.

Context precision means:
Were the retrieved contexts that are useful for answering the question ranked early?

You must evaluate each retrieved context in order.

=== Question ===
{question}

=== Retrieved Contexts in Ranking Order ===
{contexts}

Evaluation rules:

For each context, decide whether it is useful for answering the exact question.

A context is useful if it contains evidence directly related to:
- the disease or biological condition in the question,
- the intervention, exposure, gene, protein, cell type, or model in the question,
- the outcome or mechanism asked in the question,
- or a direct result/conclusion needed to answer the question.

A context is NOT useful if:
- it is only general background,
- it mentions the broad topic but not the specific question,
- it is about a different disease, molecule, cell line, model, or mechanism,
- it is only methodological and does not help answer the question,
- it is irrelevant biomedical text from another study.

Use binary verdicts:
- 1 = useful for answering the question
- 0 = not useful for answering the question

Steps:
1. Assign verdict 1 or 0 to each context.
2. Compute precision at each rank where verdict = 1.
3. Average these precision values.
4. If there are no useful contexts, score = 0.

Before scoring, think step by step:
- For each chunk, state whether it is useful and why.
- Note which useful chunks appear early vs. late.
- Compute the precision@k mentally.
- Based on this reasoning, decide the score.

Return ONLY valid compact JSON.
Do NOT use markdown.
Do NOT wrap the answer in ```json.
Do NOT use bullet points.
Do NOT use multiline explanations.
The reasoning and explanation must be short single-line strings.

Return exactly this format:
{{
  "reasoning": "<your step-by-step reasoning in one line>",
  "score": <float between 0 and 1>,
  "verdicts": [0 or 1 for each context],
  "explanation": "<short single-line explanation>"
}}
"""

print("JUDGE_PROMPT_CONTEXT_PRECISION defined.")

In [ ]:
JUDGE_PROMPT_CONTEXT_SUFFICIENCY = """\
You are an expert biomedical evaluator specialized in RAG retrieval evaluation.

Your task is to evaluate CONTEXT SUFFICIENCY.

Context sufficiency means:
Do the retrieved contexts contain enough information to answer the question correctly?

You must judge ONLY the retrieved contexts.
Do NOT use outside biomedical knowledge.

=== Question ===
{question}

=== Retrieved Contexts ===
{contexts}

Evaluation rules:
1. Determine the key elements required to answer the question.
2. Check whether the retrieved contexts contain evidence for these key elements.
3. For biomedical yes/no questions, the context is sufficient only if it supports the yes/no conclusion.
4. If the question asks about a specific mechanism, the context must mention or clearly support that mechanism.
5. If the context only provides background but not results, the score should be low.
6. If the context contains some relevant entities but does not allow a correct final answer, the score should be moderate or low.
7. If the context contains irrelevant chunks but also enough clear evidence to answer the question, the score can still be high.
8. Do NOT penalize because the generated answer is bad. This metric evaluates only whether the retrieved contexts are sufficient.

Before scoring, think step by step:
- List the key elements needed to answer the question (disease, mechanism, outcome, yes/no conclusion, etc.).
- For each element, check whether the contexts contain sufficient evidence.
- Note any important missing evidence.
- Based on this reasoning, decide the score.

Return ONLY valid compact JSON.
Do NOT use markdown.
Do NOT wrap the answer in ```json.
Do NOT use bullet points.
Do NOT use multiline explanations.
The reasoning and explanation must be short single-line strings.

Return exactly this format:
{{
  "reasoning": "<your step-by-step reasoning in one line>",
  "score": <float between 0 and 1>,
  "explanation": "<short single-line explanation>"
}}
"""

print("JUDGE_PROMPT_CONTEXT_SUFFICIENCY defined.")

In [ ]:
import json
import re

def parse_json_score(raw):
    """
    Robust score parser for judge outputs.
    Handles:
    - ```json markdown fences
    - invalid multiline explanations
    - extra text before/after JSON
    - fallback score extraction
    """

    if raw is None:
        print("[PARSE WARNING] Empty response.")
        return None

    text = str(raw).strip()

    # Remove markdown code fences
    text = re.sub(r"```json", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```", "", text).strip()

    # First: try to extract score directly, even if JSON is broken
    score_match = re.search(r'"score"\s*:\s*([0-9]*\.?[0-9]+)', text)

    if score_match:
        score = float(score_match.group(1))

        # Safety clamp between 0 and 1
        score = max(0.0, min(1.0, score))
        return score

    # Second: try normal JSON parsing if direct score extraction failed
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
            score = float(data["score"])
            score = max(0.0, min(1.0, score))
            return score
        except Exception as e:
            print(f"[PARSE WARNING] Could not parse JSON: {e}")
            print("Raw first 200 chars:", text[:200])
            return None

    print("[PARSE WARNING] No score found.")
    print("Raw first 200 chars:", text[:200])
    return None

In [ ]:
@dataclass
class CustomFaithfulness(SingleTurnMetric):
    """
    Faithfulness: checks whether the generated answer is supported by the retrieved contexts
    and whether the final conclusion correctly follows from the retrieved contexts.
    """
    name: str = "faithfulness"
    sleep_seconds: int = 60
    max_attempts: int = 5

    _required_columns: dict = field(default_factory=lambda: {
        MetricType.SINGLE_TURN: {
            "user_input", "response", "retrieved_contexts"
        }
    })

    def init(self, run_config=None):
        pass

    async def _single_turn_ascore(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:

        # Wait before calling NVIDIA, to reduce 429 errors
        await asyncio.sleep(self.sleep_seconds)

        contexts = "\n\n".join(sample.retrieved_contexts or [])

        prompt = JUDGE_PROMPT_FAITHFULNESS.format(
            question=sample.user_input,
            contexts=contexts,
            answer=sample.response,
        )

        for attempt in range(1, self.max_attempts + 1):
            try:
                response = nvidia_llm.invoke(prompt)

                raw = response.content if hasattr(response, "content") else str(response)
                raw = str(raw).strip()
                print(f"  [DEBUG] content: {repr(response.content)}")
                print(f"  [DEBUG] additional_kwargs: {response.additional_kwargs}")
                print(f"  [DEBUG] response_metadata: {response.response_metadata}")

                if raw == "":
                    wait = 60 * attempt
                    print(
                        f"  [Faithfulness] empty response. "
                        f"Attempt {attempt}/{self.max_attempts}. Waiting {wait}s..."
                    )
                    await asyncio.sleep(wait)
                    continue

                return parse_json_score(raw)

            except Exception as e:
                error_text = str(e)

                if "429" in error_text or "Too Many Requests" in error_text:
                    wait = 60 * attempt
                    print(
                        f"  [Faithfulness] 429 Too Many Requests. "
                        f"Attempt {attempt}/{self.max_attempts}. Waiting {wait}s..."
                    )
                    await asyncio.sleep(wait)
                    continue

                print(f"  [Faithfulness] error: {e}")
                raise e

        raise RuntimeError(
            "STOPPED: Faithfulness failed after repeated 429 or empty responses."
        )

    def _single_turn_score(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:
        return asyncio.get_event_loop().run_until_complete(
            self._single_turn_ascore(sample, callbacks)
        )

In [ ]:
@dataclass
class CustomContextPrecision(SingleTurnMetric):
    """
    Context Precision: were retrieved chunks useful and ranked correctly?
    Formula: mean Precision@k at relevant ranks
    """
    name: str = "context_precision"
    sleep_seconds: int = 60
    max_attempts: int = 5

    _required_columns: dict = field(default_factory=lambda: {
        MetricType.SINGLE_TURN: {
            "user_input", "retrieved_contexts"
        }
    })

    def init(self, run_config=None):
        pass

    async def _single_turn_ascore(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:

        # Wait before calling NVIDIA, to reduce 429 errors
        await asyncio.sleep(self.sleep_seconds)

        chunks_text = "\n\n".join([
            f"Chunk {i+1}: {chunk}"
            for i, chunk in enumerate(sample.retrieved_contexts or [])
        ])

        prompt = JUDGE_PROMPT_CONTEXT_PRECISION.format(
            question=sample.user_input,
            contexts=chunks_text,
        )

        for attempt in range(1, self.max_attempts + 1):
            try:
                response = nvidia_llm.invoke(prompt)

                raw = response.content if hasattr(response, "content") else str(response)
                raw = str(raw).strip()
                print(f"  [DEBUG] content: {repr(response.content)}")
                print(f"  [DEBUG] additional_kwargs: {response.additional_kwargs}")
                print(f"  [DEBUG] response_metadata: {response.response_metadata}")

                if raw == "":
                    wait = 60 * attempt
                    print(
                        f"  [ContextPrecision] empty response. "
                        f"Attempt {attempt}/{self.max_attempts}. Waiting {wait}s..."
                    )
                    await asyncio.sleep(wait)
                    continue

                return parse_json_score(raw)

            except Exception as e:
                error_text = str(e)

                if "429" in error_text or "Too Many Requests" in error_text:
                    wait = 60 * attempt
                    print(
                        f"  [ContextPrecision] 429 Too Many Requests. "
                        f"Attempt {attempt}/{self.max_attempts}. Waiting {wait}s..."
                    )
                    await asyncio.sleep(wait)
                    continue

                print(f"  [ContextPrecision] error: {e}")
                raise e

        raise RuntimeError(
            "STOPPED: ContextPrecision failed after repeated 429 or empty responses."
        )

    def _single_turn_score(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:
        return asyncio.get_event_loop().run_until_complete(
            self._single_turn_ascore(sample, callbacks)
        )

print("CustomContextPrecision defined.")

In [ ]:
@dataclass
class CustomAnswerRelevancy(SingleTurnMetric):
    name: str = "answer_relevancy"
    sleep_seconds: int = 60
    max_attempts: int = 5

    _required_columns: dict = field(default_factory=lambda: {
        MetricType.SINGLE_TURN: {
            "user_input", "response"
        }
    })

    def init(self, run_config=None):
        pass

    async def _single_turn_ascore(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:

        # Wait before calling NVIDIA, to reduce 429 errors
        await asyncio.sleep(self.sleep_seconds)

        prompt = JUDGE_PROMPT_ANSWER_RELEVANCY.format(
            question=sample.user_input,
            answer=sample.response,
        )

        for attempt in range(1, self.max_attempts + 1):
            try:
                response = nvidia_llm.invoke(prompt)

                raw = response.content if hasattr(response, "content") else str(response)
                raw = str(raw).strip()
                print(f"  [DEBUG] content: {repr(response.content)}")
                print(f"  [DEBUG] additional_kwargs: {response.additional_kwargs}")
                print(f"  [DEBUG] response_metadata: {response.response_metadata}")

                if raw == "":
                    wait = 60 * attempt
                    print(
                        f"  [AnswerRelevancy] empty response. "
                        f"Attempt {attempt}/{self.max_attempts}. Waiting {wait}s..."
                    )
                    await asyncio.sleep(wait)
                    continue

                return parse_json_score(raw)

            except Exception as e:
                error_text = str(e)

                if "429" in error_text or "Too Many Requests" in error_text:
                    wait = 60 * attempt
                    print(
                        f"  [AnswerRelevancy] 429 Too Many Requests. "
                        f"Attempt {attempt}/{self.max_attempts}. Waiting {wait}s..."
                    )
                    await asyncio.sleep(wait)
                    continue

                print(f"  [AnswerRelevancy] error: {e}")
                raise e

        raise RuntimeError(
            "STOPPED: AnswerRelevancy failed after repeated 429 or empty responses."
        )

    def _single_turn_score(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:
        return asyncio.get_event_loop().run_until_complete(
            self._single_turn_ascore(sample, callbacks)
        )

print("CustomAnswerRelevancy defined.")

In [ ]:
@dataclass
class CustomContextSufficiency(SingleTurnMetric):
    name: str = "context_sufficiency"
    sleep_seconds: int = 60
    max_attempts: int = 5

    _required_columns: dict = field(default_factory=lambda: {
        MetricType.SINGLE_TURN: {
            "user_input", "retrieved_contexts"
        }
    })

    def init(self, run_config=None):
        pass

    async def _single_turn_ascore(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:

        # Wait before calling NVIDIA, to reduce 429 errors
        await asyncio.sleep(self.sleep_seconds)

        contexts = "\n\n".join([
            f"Chunk {i+1}: {chunk}"
            for i, chunk in enumerate(sample.retrieved_contexts or [])
        ])

        prompt = JUDGE_PROMPT_CONTEXT_SUFFICIENCY.format(
            question=sample.user_input,
            contexts=contexts,
        )

        for attempt in range(1, self.max_attempts + 1):
            try:
                response = nvidia_llm.invoke(prompt)

                raw = response.content if hasattr(response, "content") else str(response)
                raw = str(raw).strip()
                print(f"  [DEBUG] content: {repr(response.content)}")
                print(f"  [DEBUG] additional_kwargs: {response.additional_kwargs}")
                print(f"  [DEBUG] response_metadata: {response.response_metadata}")

                if raw == "":
                    wait = 60 * attempt
                    print(
                        f"  [ContextSufficiency] empty response. "
                        f"Attempt {attempt}/{self.max_attempts}. Waiting {wait}s..."
                    )
                    await asyncio.sleep(wait)
                    continue

                return parse_json_score(raw)

            except Exception as e:
                error_text = str(e)

                if "429" in error_text or "Too Many Requests" in error_text:
                    wait = 60 * attempt
                    print(
                        f"  [ContextSufficiency] 429 Too Many Requests. "
                        f"Attempt {attempt}/{self.max_attempts}. Waiting {wait}s..."
                    )
                    await asyncio.sleep(wait)
                    continue

                print(f"  [ContextSufficiency] error: {e}")
                raise e

        raise RuntimeError(
            "STOPPED: ContextSufficiency failed after repeated 429 or empty responses."
        )

    def _single_turn_score(
        self, sample: SingleTurnSample, callbacks=None
    ) -> float:
        return asyncio.get_event_loop().run_until_complete(
            self._single_turn_ascore(sample, callbacks)
        )

print("CustomContextSufficiency defined.")

In [ ]:
print("\nInstantiating metrics...")

custom_faithfulness = CustomFaithfulness()
print(" 1. CustomFaithfulness     — binary YES/NO, 1 call/sample")

#print("\n   Building AnswerRelevancy...")
#standard_answer_relevancy = build_answer_relevancy_metric(
#    judge_llm        = judge_llm,
#    judge_embeddings = judge_embeddings,
#)
#print(" 2. AnswerRelevancy         — standard Ragas cosine similarity")
custom_answer_relevancy = CustomAnswerRelevancy()  # ← NEW
print(" 2. CustomAnswerRelevancy — direct LLM scoring, 1 call/sample")

#print("\n   Building ContextPrecision...")
#standard_context_precision = build_context_precision_metric(
#    judge_llm = judge_llm,
#)
#print(" 3. ContextPrecision        — standard Ragas Precision@k")
custom_context_precision = CustomContextPrecision()
print("3. CustomContextPrecision  — Precision@k, 1 call/sample")

custom_context_sufficiency = CustomContextSufficiency()
print("4. CustomContextSufficiency  — answerability, 1 call, no ground truth")

metrics = [
    custom_faithfulness,
    #standard_answer_relevancy,
    custom_answer_relevancy, 
    custom_context_precision,
    #standard_context_precision,
    custom_context_sufficiency,
]

print("\n All 4 metrics ready.")
print("\n   Formula summary:")
print("   faithfulness          → answer grounded in context?")
print("   answer_relevancy      → answer addresses question?")
print("   context_precision     → chunks useful and ranked?")
print("   context_sufficiency   → context enough to answer?")

In [ ]:
run_config = RunConfig(
    timeout     = 600,
    max_workers = 1, 
    max_retries = 3,
)

print(" RunConfig ready.")
print(f"   timeout     : {run_config.timeout}s")
print(f"   max_workers : {run_config.max_workers} (sequential)")
print(f"   max_retries : {run_config.max_retries}")


In [ ]:
# TEST RUN ON 1 SAMPLE TO CHECK CONNECTIVITY
print("\n--- Connectivity test (1 sample) ---")

test_sample = SingleTurnSample(
    user_input         = combined_df.iloc[0]["question"],
    response           = combined_df.iloc[0]["generated_answer"],
    retrieved_contexts = combined_df.iloc[0]["retrieved_contexts"],
    reference          = combined_df.iloc[0]["ground_truth"],
)

test_dataset = EvaluationDataset(samples=[test_sample])
test_result  = evaluate(
    dataset    = test_dataset,
    metrics    = metrics,
    run_config = run_config,
)

print("\n Connectivity test passed. Sample scores:")
print(test_result)
print("\nIf all scores are non-zero real decimals → proceed to Cell 15.")


In [ ]:
def evaluate_model_with_ragas(
    model_key  : str,
    model_df   : pd.DataFrame,
    metrics    : list,
    run_config : RunConfig,
    batch_size : int = 5,
    sleep_s    : int = 90,
) -> pd.DataFrame:
    """
    Evaluates all rows for one generator model using custom Ragas metrics.

    Folder structure created:
       results/RAGASOutput/{model_key}/
           batch_01_rows_0_to_4.csv     ← checkpoint after each batch
           batch_02_rows_5_to_9.csv
           ...
           ragas_{model_key}.csv        ← final merged file

    Resume logic:
       If batch file already exists → skip it
       Continues from where it stopped after a crash
    """
    # ── Create model subfolder ────────────────────────────────────────────────
    model_dir = os.path.join(CONFIG["ragas_output_dir"], model_key)
    os.makedirs(model_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  Evaluating : {model_key}")
    print(f"  Samples    : {len(model_df)} questions")
    print(f"  Batch size : {batch_size} samples per batch")
    print(f"  Sleep      : {sleep_s}s between batches")
    print(f"  Output dir : {model_dir}/")
    print(f"{'='*60}")

    num_batches = (len(model_df) + batch_size - 1) // batch_size
    all_scores  = []

    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx   = min(start_idx + batch_size, len(model_df))
        batch_df  = model_df.iloc[start_idx:end_idx]

        # ── Batch file path ───────────────────────────────────────────────────
        batch_path = os.path.join(
            model_dir,
            f"batch_{batch_idx+1:02d}_rows_{start_idx}_to_{end_idx-1}.csv"
        )

        # ── Resume: skip if already saved ─────────────────────────────────────
        if os.path.exists(batch_path):
            existing = pd.read_csv(batch_path)
            all_scores.append(existing)
            print(f"\n  Batch {batch_idx+1}/{num_batches} "
                  f"(rows {start_idx}-{end_idx-1}) "
                  f"→ Already done — loaded from checkpoint")
            continue

        print(f"\n  Batch {batch_idx+1}/{num_batches} "
              f"(rows {start_idx} to {end_idx-1})...")

        samples = [
            SingleTurnSample(
                user_input         = row["question"],
                response           = row["generated_answer"],
                retrieved_contexts = row["retrieved_contexts"],
            )
            for _, row in batch_df.iterrows()
        ]

        dataset = EvaluationDataset(samples=samples)

        try:
            result    = evaluate(
                dataset    = dataset,
                metrics    = metrics,
                run_config = run_config,
            )
            scores_df = result.to_pandas()
            scores_df.insert(0, "model_key",   model_key)
            scores_df.insert(1, "question_id",
                             batch_df["question_id"].values)
            scores_df = scores_df.rename(columns={
                "user_input": "question",
                "response"  : "generated_answer",
            })

            # ── Save batch checkpoint ─────────────────────────────────────────
            scores_df.to_csv(batch_path, index=False)
            all_scores.append(scores_df)

            faith = scores_df["faithfulness"].mean()
            ans   = scores_df["answer_relevancy"].mean()
            prec  = scores_df["context_precision"].mean()
            suff  = scores_df["context_sufficiency"].mean()
            print(f"   Batch done — "
                  f"faith={faith:.3f}  ans={ans:.3f}  "
                  f"prec={prec:.3f}  suff={suff:.3f}")  
            print(f"   Saved → {batch_path}")

        except Exception as e:
            error_text = str(e)

            print(f"   Batch {batch_idx+1} failed: {e}")
            print(f"    This batch was not saved and will be retried on next run.")

            if "429" in error_text or "Too Many Requests" in error_text:
                print("\n   NVIDIA 429 Too Many Requests detected.")
                print("  Stop evaluation now to avoid sending more requests.")
                print("  Wait a few minutes, then rerun the same cell.")
                print("  Completed batches will be loaded from checkpoint.")
                raise e

            raise e
        if batch_idx < num_batches - 1:
            print(f"  Waiting {sleep_s}s before next batch...")
            time.sleep(sleep_s)

    # ── Merge all batches → final CSV in model folder ─────────────────────────
    if not all_scores:
        print(f"\n No results for {model_key}")
        return pd.DataFrame()

    final_df   = pd.concat(all_scores, ignore_index=True)
    final_path = os.path.join(model_dir, f"ragas_{model_key}.csv")
    final_df.to_csv(final_path, index=False)

    print(f"\n   Done — {model_key}")
    print(f"     faithfulness       : {final_df['faithfulness'].mean():.3f}")
    print(f"     answer_relevancy   : {final_df['answer_relevancy'].mean():.3f}")
    print(f"     context_precision  : {final_df['context_precision'].mean():.3f}")
    print(f"     context_sufficiency: {final_df['context_sufficiency'].mean():.3f}")
    print(f"     Batches saved in   : {model_dir}/")
    print(f"     Final CSV          : {final_path}")

    return final_df

print("evaluate_model_with_ragas() with checkpoint saving ready.")

In [ ]:
print("\nInitializing Judge LLM...")

nvidia_llm = ChatNVIDIA(
    model=CONFIG["judge_model"],
    temperature=CONFIG["judge_temperature"],
    max_completion_tokens=CONFIG["judge_max_completion_tokens"],
)

judge_llm = LangchainLLMWrapper(nvidia_llm)

print(f"Judge LLM initialized: {CONFIG['judge_model']}")

##### **Judge sur 50 les questions**

In [ ]:
# FULL EVALUATION — llama_3b
TARGET_MODEL = "llama_3b"
model_df     = combined_df[
    combined_df["model_key"] == TARGET_MODEL
].reset_index(drop=True)
print(f"Model: {TARGET_MODEL} | Questions: {len(model_df)}")

scores_df = evaluate_model_with_ragas(
    model_key  = TARGET_MODEL,
    model_df   = model_df,
    metrics    = metrics,
    run_config = run_config,
    batch_size = CONFIG["batch_size"],
    sleep_s    = CONFIG["sleep_between_batches"],
)

print(f"\n Done — results/RAGASOutput/{TARGET_MODEL}/ragas_{TARGET_MODEL}.csv")

In [ ]:
# FULL EVALUATION — llama_8b
TARGET_MODEL = "llama_8b"
model_df     = combined_df[
    combined_df["model_key"] == TARGET_MODEL
].reset_index(drop=True)
print(f"Model: {TARGET_MODEL} | Questions: {len(model_df)}")

scores_df = evaluate_model_with_ragas(
    model_key  = TARGET_MODEL,
    model_df   = model_df,
    metrics    = metrics,
    run_config = run_config,
    batch_size = CONFIG["batch_size"],
    sleep_s    = CONFIG["sleep_between_batches"],
)

print(f"\n Done — results/RAGASOutput/{TARGET_MODEL}/ragas_{TARGET_MODEL}.csv")

In [ ]:
# FULL EVALUATION — mistral_small
TARGET_MODEL = "mistral_small"
model_df     = combined_df[
    combined_df["model_key"] == TARGET_MODEL
].reset_index(drop=True)
print(f"Model: {TARGET_MODEL} | Questions: {len(model_df)}")

scores_df = evaluate_model_with_ragas(
    model_key  = TARGET_MODEL,
    model_df   = model_df,
    metrics    = metrics,
    run_config = run_config,
    batch_size = CONFIG["batch_size"],
    sleep_s    = CONFIG["sleep_between_batches"],
)

print(f"\n Done — results/RAGASOutput/{TARGET_MODEL}/ragas_{TARGET_MODEL}.csv")

In [ ]:
# FULL EVALUATION — llama_70b
TARGET_MODEL = "llama_70b"
model_df     = combined_df[
    combined_df["model_key"] == TARGET_MODEL
].reset_index(drop=True)
print(f"Model: {TARGET_MODEL} | Questions: {len(model_df)}")

scores_df = evaluate_model_with_ragas(
    model_key  = TARGET_MODEL,
    model_df   = model_df,
    metrics    = metrics,
    run_config = run_config,
    batch_size = CONFIG["batch_size"],
    sleep_s    = CONFIG["sleep_between_batches"],
)

print(f"\n Done — results/RAGASOutput/{TARGET_MODEL}/ragas_{TARGET_MODEL}.csv")

In [ ]:
# FULL EVALUATION — qwen_397b
TARGET_MODEL = "qwen_397b"
model_df     = combined_df[
    combined_df["model_key"] == TARGET_MODEL
].reset_index(drop=True)
print(f"Model: {TARGET_MODEL} | Questions: {len(model_df)}")

scores_df = evaluate_model_with_ragas(
    model_key  = TARGET_MODEL,
    model_df   = model_df,
    metrics    = metrics,
    run_config = run_config,
    batch_size = CONFIG["batch_size"],
    sleep_s    = CONFIG["sleep_between_batches"],
)

print(f"\n Done — results/RAGASOutput/{TARGET_MODEL}/ragas_{TARGET_MODEL}.csv")

In [ ]:
# COMBINE ALL 5 RAGAS RESULTS
# Reads final CSV from each model's subfolder:
#   results/RAGASOutput/{model_key}/ragas_{model_key}.csv
# Merges into one combined file:
#   results/RAGASOutput/ragas_judge_results.csv

print("\nCombining all model RAGAS results...")

RAGAS_OUTPUT_DIR = "results/RAGASOutput"

MODEL_KEYS = list(CONFIG["generator_models"].keys())
all_dfs    = []

for model_key in MODEL_KEYS:

    # Read final CSV from model subfolder
    path = os.path.join(
        RAGAS_OUTPUT_DIR,
        model_key,
        f"ragas_{model_key}.csv"
    )

    if os.path.exists(path):
        df = pd.read_csv(path)
        all_dfs.append(df)
        print(f" {model_key}: {len(df)} rows")
    else:
        print(f"  Not found: {path}")

if not all_dfs:
    print(" No files found — run all 5 evaluation cells first")
else:
    judge_df   = pd.concat(all_dfs, ignore_index=True)

    # Save combined in RAGASOutput/ root
    judge_path = os.path.join(
        RAGAS_OUTPUT_DIR,
        "ragas_judge_results.csv"
    )
    judge_df.to_csv(judge_path, index=False)

    print(f"\n Combined: {len(judge_df)} rows")
    print(f"   Models  : {judge_df['model_key'].unique().tolist()}")
    print(f"   Saved   : {judge_path}")

In [ ]:
print("\nBuilding final summary table...")

score_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_sufficiency",
]

judge_summary = (
    judge_df
    .groupby("model_key")[score_cols]
    .mean()
    .round(4)
    .reset_index()
)

judge_summary["overall_score"] = (
    judge_summary[score_cols].mean(axis=1).round(4)
)

latency = (
    combined_df
    .groupby("model_key")["inference_time_sec"]
    .mean()
    .round(3)
    .reset_index()
    .rename(columns={"inference_time_sec": "avg_latency_sec"})
)

final_summary = pd.merge(
    judge_summary, latency, on="model_key", how="left"
)
final_summary = final_summary.sort_values(
    "overall_score", ascending=False
)

RAGAS_OUTPUT_DIR = "results/RAGASOutput"
os.makedirs(RAGAS_OUTPUT_DIR, exist_ok=True)

summary_path = os.path.join(
    RAGAS_OUTPUT_DIR, "ragas_judge_summary.csv"
)
final_summary.to_csv(summary_path, index=False)

print("\n" + "="*70)
print("  FINAL BENCHMARKING RESULTS")
print("="*70)
print(final_summary.to_string(index=False))

print("\n" + "="*70)
print("  METRIC INTERPRETATION")
print("="*70)
print("  All scores 0.0–1.0. Higher is better.")
print()
print("   faithfulness        → Answer grounded in context")
print("                         Binary supported/total")
print("   answer_relevancy    → Answer addresses question")
print("                         Needs-based, judge scores")
print("   context_precision   → Chunks useful and ranked")
print("                         Standard Precision@k")
print("   context_sufficiency → Context sufficient to answer")
print("                         Reference-free, judge scores")
print("  overall_score     → Mean of 4 metrics")
print("  avg_latency_sec   → Phase 1 generation time")

print(f"\n Phase 2 complete.")
print(f"   → {judge_path}")
print(f"   → {summary_path}")

#files.download(judge_path)
#files.download(summary_path)
print(" Final files downloaded.")


### **Visualizations**

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

RAGAS_OUTPUT_DIR = "results/RAGASOutput"
VIZ_DIR = os.path.join(RAGAS_OUTPUT_DIR, "visualizations")
os.makedirs(VIZ_DIR, exist_ok=True)

print(f"Visualizations will be saved in: {VIZ_DIR}")

In [ ]:
judge_path = os.path.join(RAGAS_OUTPUT_DIR, "ragas_judge_results.csv")
summary_path = os.path.join(RAGAS_OUTPUT_DIR, "ragas_judge_summary.csv")

judge_df = pd.read_csv(judge_path)
final_summary = pd.read_csv(summary_path)

print("judge_df:", judge_df.shape)
print("final_summary:", final_summary.shape)

display(final_summary)

In [ ]:
plot_df = final_summary.sort_values("overall_score", ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(plot_df["model_key"], plot_df["overall_score"])
plt.xlabel("Overall score")
plt.ylabel("Model")
plt.title("Overall RAGAS Judge Score by Model")
plt.xlim(0, 1)

for i, value in enumerate(plot_df["overall_score"]):
    plt.text(value + 0.01, i, f"{value:.3f}", va="center")

plt.tight_layout()

path = os.path.join(VIZ_DIR, "overall_score_by_model.png")
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")

In [ ]:
score_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_sufficiency",
]

plot_df = final_summary.sort_values("overall_score", ascending=False)

x = np.arange(len(plot_df["model_key"]))
width = 0.2

plt.figure(figsize=(12, 6))

for i, col in enumerate(score_cols):
    plt.bar(x + i * width, plot_df[col], width, label=col)

plt.xticks(x + width * 1.5, plot_df["model_key"], rotation=30, ha="right")
plt.ylabel("Score")
plt.title("Comparison of RAGAS Metrics by Model")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()

path = os.path.join(VIZ_DIR, "metric_comparison_by_model.png")
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    final_summary["avg_latency_sec"],
    final_summary["overall_score"],
    s=100
)

for _, row in final_summary.iterrows():
    plt.text(
        row["avg_latency_sec"] + 0.02,
        row["overall_score"],
        row["model_key"],
        fontsize=9
    )

plt.xlabel("Average latency in seconds")
plt.ylabel("Overall score")
plt.title("Quality vs Latency")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()

path = os.path.join(VIZ_DIR, "quality_vs_latency.png")
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")

In [ ]:
heatmap_df = final_summary.set_index("model_key")[score_cols]

plt.figure(figsize=(9, 5))
plt.imshow(heatmap_df.values, aspect="auto")

plt.xticks(np.arange(len(score_cols)), score_cols, rotation=30, ha="right")
plt.yticks(np.arange(len(heatmap_df.index)), heatmap_df.index)

for i in range(heatmap_df.shape[0]):
    for j in range(heatmap_df.shape[1]):
        plt.text(
            j, i,
            f"{heatmap_df.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.colorbar(label="Score")
plt.title("Heatmap of RAGAS Metrics by Model")
plt.tight_layout()

path = os.path.join(VIZ_DIR, "metric_heatmap.png")
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")

In [ ]:
judge_df["row_overall_score"] = judge_df[score_cols].mean(axis=1)

# If question_id exists, use it. Otherwise use question text.
question_col = "question_id" if "question_id" in judge_df.columns else "question"

best_per_question = (
    judge_df
    .sort_values("row_overall_score", ascending=False)
    .groupby(question_col)
    .first()
    .reset_index()
)

winner_counts = (
    best_per_question["model_key"]
    .value_counts()
    .reset_index()
)

winner_counts.columns = ["model_key", "num_best_answers"]

plot_df = winner_counts.sort_values("num_best_answers", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(plot_df["model_key"], plot_df["num_best_answers"])
plt.xlabel("Number of questions where model is best")
plt.ylabel("Model")
plt.title("Best Model Count Across Questions")

for i, value in enumerate(plot_df["num_best_answers"]):
    plt.text(value + 0.1, i, str(value), va="center")

plt.tight_layout()

path = os.path.join(VIZ_DIR, "best_model_count.png")
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")

display(winner_counts)

In [ ]:
# Save final summary again
final_summary.to_csv(
    os.path.join(RAGAS_OUTPUT_DIR, "ragas_judge_summary.csv"),
    index=False
)

# Save per-question best model table
best_per_question.to_csv(
    os.path.join(RAGAS_OUTPUT_DIR, "best_model_per_question.csv"),
    index=False
)

# Save winner counts
winner_counts.to_csv(
    os.path.join(RAGAS_OUTPUT_DIR, "best_model_counts.csv"),
    index=False
)

print("Saved:")
print(os.path.join(RAGAS_OUTPUT_DIR, "ragas_judge_summary.csv"))
print(os.path.join(RAGAS_OUTPUT_DIR, "best_model_per_question.csv"))
print(os.path.join(RAGAS_OUTPUT_DIR, "best_model_counts.csv"))